# Medical Consultation Platform — Visual Analysis v2

## Data Sources

| File | Location | Used for |
|---|---|---|
| `crm_stage_finished.xlsx` | `data/processed/` | All financial charts — Topics 0,1,2,4,5,6,7,8.1/8.4/8.5. 3,940 completed records with EUR-normalised columns and Gross profit pre-computed |
| `crm_selected_translated_column.xlsx` | `data/processed/` | Volume, cancellation, cohort, new/returning patient charts — Topics 3, 8.2/8.3/8.6. All 9,974 records including Cancelled |
| `doctor_specialty_group.xlsx` | `data/raw/hierarchy/` | English specialty groupings joined in Topic 7 charts |
| `fx_rate_eur.xlsx` | `data/raw/fx_rate/` | Daily EUR FX rates used in Topic 6 (RUB/EUR rate vs GP trend) |
| `origin_CRM_2026_03_29.xlsx` | `data/raw/original_CRM/` | **Not used** — Sergey's pipeline replaced it entirely |

## Notes
- **Tariff price** (`Patient Price, EUR`): the pre-agreed fee for a consultation set in the CRM at booking time, denominated in EUR. This is what *should* be charged — it differs from actual patient payment received (`Patient payment, EUR`) due to cancellations, discounts, and partial payments.
- **Outlier strategy:** clip at p99 for histogram/boxplot/scatter; keep raw values for Pareto and aggregated bar charts.
- **Dual output:** Matplotlib inline in notebook (visible on GitHub after saving with outputs) + Plotly charts captured silently → exported to `visualisations/quick_viz_4.html`

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams.update({
    'figure.figsize': (11, 4),
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'sans-serif',
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
})

BASE_DIR = BASE_DIR if 'BASE_DIR' in vars() else Path().resolve()
PROCESSED_DIR = BASE_DIR / 'data' / 'processed'
HIERARCHY_DIR = BASE_DIR / 'data' / 'raw' / 'hierarchy'
VIZ_DIR       = BASE_DIR / 'visualisations'
VIZ_DIR.mkdir(parents=True, exist_ok=True)

# ---- LOAD FILES --------
df_done = pd.read_excel(PROCESSED_DIR / 'crm_stage_finished.xlsx')
df_all  = pd.read_excel(PROCESSED_DIR / 'crm_selected_translated_column.xlsx')
spec_grp = pd.read_excel(HIERARCHY_DIR / 'doctor_specialty_group.xlsx')

# ---- DATE & MONTH COLUMNS --------
df_done['Month']         = pd.to_datetime(df_done['Created Date']).dt.to_period('M').astype(str)
df_done['CalendarMonth'] = pd.to_datetime(df_done['Created Date']).dt.month
df_all['Month']          = pd.to_datetime(df_all['Created Date']).dt.to_period('M').astype(str)

# ---- SPECIALTY JOIN --------
df_done['Specialty_primary'] = df_done['Doctor Specialty'].str.split(';').str[0].str.strip()
df_done = df_done.merge(
    spec_grp.rename(columns={'Doctor Specialty': 'Specialty_primary'}),
    on='Specialty_primary', how='left'
)

# ---- USDT → GROUP WITH USD --------
df_done['Patient Payment Currency'] = df_done['Patient Payment Currency'].replace('USDT','USD')
df_done['Doctor Payment Currency 1'] = df_done['Doctor Payment Currency 1'].replace('USDT','USD')

# ---- SUBSETS --------
CONSULTANCY   = ['Video Consultancy', 'Chat Consultancy']
df_consult    = df_done[df_done['Service Type Group'].isin(CONSULTANCY)].copy()
df_consult_all= df_all[df_all['Service Type Group'].isin(CONSULTANCY)].copy()

# ---- OUTLIER CLIPS (p99 per key financial column) --------
P99_PRICE = df_done['Patient Price, EUR'].quantile(0.99)
P99_COST  = df_done['Doctor cost, EUR'].quantile(0.99)
P01_GP    = df_done['Gross profit'].quantile(0.01)
P99_GP    = df_done['Gross profit'].quantile(0.99)

# ---- PALETTE --------
PALETTE = {
    'Video Consultancy':'#2E86AB','Chat Consultancy':'#A23B72',
    'BU Consultancy':'#2E86AB','BU others':'#F18F01',
    'Closed':'#27AE60','Cancelled':'#E74C3C','Stage others':'#F39C12',
    '':'#2E86AB','RUB':'#E74C3C','USD':'#F18F01','ILS':'#27AE60',
}

# ---- HTML COLLECTOR --------
HTML_DIVS = []

def mpl_show(fig):
    plt.tight_layout()
    plt.show()
    plt.close(fig)

def plotly_capture(fig, title):
    fig.update_layout(title=dict(text=title, font=dict(size=13)),
                      template='plotly_white',
                      margin=dict(t=55, b=35, l=40, r=40))
    include_js = 'cdn' if not HTML_DIVS else False
    HTML_DIVS.append((title, fig.to_html(full_html=False, include_plotlyjs=include_js)))

print(f'Completed records:  {len(df_done):,}')
print(f'All records:        {len(df_all):,}')
print(f'Date range:         {df_done["Month"].min()} → {df_done["Month"].max()}')
print(f'p99 Price EUR:      {P99_PRICE:.0f}  |  p99 Cost EUR: {P99_COST:.0f}  |  GP p01/p99: {P01_GP:.0f}/{P99_GP:.0f}')

---
## Topic 0 — Business Overview
### Chart 0.1 · Monthly Revenue + Gross Profit (dual axis)

In [ ]:
monthly_rev = df_done.groupby('Month').agg(
    Revenue=('Patient Price, EUR','sum'),
    GP=('Gross profit','sum')
).reset_index()

# ---- Matplotlib --------
fig, ax1 = plt.subplots()
ax1.bar(monthly_rev['Month'], monthly_rev['Revenue'], color='#2E86AB', alpha=0.6, label='Revenue EUR')
ax1.set_ylabel('Revenue EUR', color='#2E86AB')
ax1.tick_params(axis='x', rotation=45)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))
ax2 = ax1.twinx()
ax2.plot(monthly_rev['Month'], monthly_rev['GP'], color='#27AE60', linewidth=2, marker='o', markersize=4, label='Gross Profit')
ax2.axhline(0, color='grey', linestyle='--', linewidth=0.8)
ax2.set_ylabel('Gross Profit EUR', color='#27AE60')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))
ax1.set_title('Monthly Revenue vs Gross Profit (EUR)')
mpl_show(fig)

# ---- Plotly --------
pf = make_subplots(specs=[[{'secondary_y':True}]])
pf.add_trace(go.Bar(x=monthly_rev['Month'], y=monthly_rev['Revenue'],
                    marker_color='#2E86AB', opacity=0.65, name='Revenue EUR'), secondary_y=False)
pf.add_trace(go.Scatter(x=monthly_rev['Month'], y=monthly_rev['GP'],
                        line=dict(color='#27AE60',width=2), marker=dict(size=5),
                        name='Gross Profit EUR'), secondary_y=True)
pf.add_hline(y=0, line_dash='dash', line_color='grey', secondary_y=True)
pf.update_yaxes(title_text='Revenue EUR', secondary_y=False)
pf.update_yaxes(title_text='Gross Profit EUR', secondary_y=True)
plotly_capture(pf, '0.1 · Monthly Revenue vs Gross Profit')

### Chart 0.2 · Monthly Gross Margin % over Time

In [ ]:
monthly_margin_pct = df_done[df_done['Patient Price, EUR'] > 0].groupby('Month').apply(
    lambda x: (x['Gross profit'].sum() / x['Patient Price, EUR'].sum() * 100)
).reset_index(name='GM%')

# ---- Matplotlib --------
fig, ax = plt.subplots()
colors = ['#27AE60' if v >= 0 else '#E74C3C' for v in monthly_margin_pct['GM%']]
ax.bar(monthly_margin_pct['Month'], monthly_margin_pct['GM%'], color=colors, alpha=0.75)
ax.axhline(0, color='grey', linestyle='--', linewidth=0.8)
ax.set_title('Monthly Gross Margin % — green = positive, red = negative')
ax.set_ylabel('Gross Margin %')
ax.tick_params(axis='x', rotation=45)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:.1f}%'))
mpl_show(fig)

# ---- Plotly --------
pf = px.bar(monthly_margin_pct, x='Month', y='GM%',
            color='GM%', color_continuous_scale=['#E74C3C','#F39C12','#27AE60'],
            labels={'GM%':'Gross Margin %','Month':''})
pf.add_hline(y=0, line_dash='dash', line_color='grey')
plotly_capture(pf, '0.2 · Monthly Gross Margin %')

### Chart 0.3 · Revenue by Currency Mix

In [ ]:
%%script false
rev_ccy = df_done.groupby('Patient Payment Currency')['Patient payment , EUR'].sum().reset_index()
rev_ccy.columns = ['Currency','Revenue EUR']
rev_ccy = rev_ccy[rev_ccy['Revenue EUR'] > 0].sort_values('Revenue EUR', ascending=False)

# ---- Matplotlib --------
fig, axes = plt.subplots(1, 2, figsize=(12,4))
rev_ccy.plot.bar(ax=axes[0], x='Currency', y='Revenue EUR',
                 color=[PALETTE.get(c,'#95A5A6') for c in rev_ccy['Currency']],
                 edgecolor='white', legend=False, rot=0)
axes[0].set_title('Revenue by Patient Currency (EUR-norm)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))
axes[1].pie(rev_ccy['Revenue EUR'], labels=rev_ccy['Currency'],
            colors=[PALETTE.get(c,'#95A5A6') for c in rev_ccy['Currency']],
            autopct='%1.1f%%', startangle=90)
axes[1].set_title('Currency Share of Revenue')
mpl_show(fig)

# ---- Plotly --------
pf = px.bar(rev_ccy, x='Currency', y='Revenue EUR',
            color='Currency', color_discrete_map=PALETTE,
            text=rev_ccy['Revenue EUR'].apply(lambda x: f'EUR{x:,.0f}'),
            labels={'Revenue EUR':'Revenue EUR-norm'})
pf.update_traces(textposition='outside')
plotly_capture(pf, '0.3 · Revenue by Currency Mix')

### Chart 0.4 · Revenue by Currency over Time (EUR-normalised)

In [ ]:
%%script false
rev_time = df_done.groupby(['Month','Patient Payment Currency'])['Patient payment , EUR'].sum().reset_index()
rev_time.columns = ['Month','Currency','Revenue EUR']

# ---- Matplotlib --------
pivot = rev_time.pivot_table(index='Month', columns='Currency', values='Revenue EUR', fill_value=0)
fig, ax = plt.subplots()
pivot.plot.area(ax=ax, color=[PALETTE.get(c,'#95A5A6') for c in pivot.columns], alpha=0.8)
ax.set_title('Revenue by Patient Currency over Time (EUR-norm)')
ax.set_ylabel('Revenue EUR')
ax.tick_params(axis='x', rotation=45)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))
mpl_show(fig)

# ---- Plotly --------
pf = px.area(rev_time, x='Month', y='Revenue EUR', color='Currency',
             color_discrete_map=PALETTE, labels={'Month':'','Revenue EUR':'Revenue EUR-norm'})
plotly_capture(pf, '0.4 · Revenue by Currency over Time')

### Chart 0.5 · Seasonal Revenue Pattern (by Calendar Month)

In [ ]:
# Seasonal Revenue Pattern — line per year, point annotated as 12.5k
import calendar

df_done['Year']          = pd.to_datetime(df_done['Created Date']).dt.year
df_done['CalendarMonth'] = pd.to_datetime(df_done['Created Date']).dt.month

seasonal_yr = (df_done.groupby(['Year','CalendarMonth'])['Patient Price, EUR']
               .sum().reset_index())
seasonal_yr.columns = ['Year','Month','Revenue']

def fmt_k(v):
    """Format as 12.5k — period decimal, k suffix."""
    return f'{v/1000:.1f}k'

month_labels = [calendar.month_abbr[m] for m in range(1,13)]
year_colors  = {2024:'#95A5A6', 2025:'#2E86AB', 2026:'#F18F01'}
year_styles  = {2024:'--', 2025:'-', 2026:'-.'}
year_markers = {2024:'s', 2025:'o', 2026:'^'}

# ---- Matplotlib --------
fig, ax = plt.subplots(figsize=(12,5))
for yr, grp in seasonal_yr.groupby('Year'):
    grp = grp.sort_values('Month')
    line = ax.plot(grp['Month'], grp['Revenue'],
                   color=year_colors[yr], linestyle=year_styles[yr],
                   linewidth=2, marker=year_markers[yr], markersize=6,
                   label=str(yr))
    for _, row in grp.iterrows():
        ax.annotate(fmt_k(row['Revenue']),
                    (row['Month'], row['Revenue']),
                    textcoords='offset points', xytext=(0, 8),
                    ha='center', fontsize=8,
                    color=year_colors[yr])
ax.set_xticks(range(1,13))
ax.set_xticklabels(month_labels)
ax.set_title('Monthly Revenue by Year — Seasonal Pattern (EUR)')
ax.set_ylabel('Revenue (EUR)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x/1000:.0f}k'))
ax.legend(title='Year')
ax.annotate('2024 & 2026 are partial years', xy=(0.01,0.02),
            xycoords='axes fraction', fontsize=8, color='#95A5A6')
mpl_show(fig)

# ---- Plotly --------
pf = go.Figure()
for yr, grp in seasonal_yr.groupby('Year'):
    grp = grp.sort_values('Month')
    pf.add_trace(go.Scatter(
        x=[month_labels[m-1] for m in grp['Month']],
        y=grp['Revenue'],
        mode='lines+markers+text',
        name=str(yr),
        text=[fmt_k(v) for v in grp['Revenue']],
        textposition='top center',
        line=dict(color=year_colors[yr], width=2),
        marker=dict(size=7)
    ))
pf.update_yaxes(tickformat='.0f')
pf.add_annotation(text='2024 & 2026 are partial years',
                  xref='paper', yref='paper', x=0.01, y=0.02,
                  showarrow=False, font=dict(size=10, color='#95A5A6'))
plotly_capture(pf, '0.5 · Seasonal Revenue by Year')

### Chart 0.6 · BU Revenue Contribution over Time

In [ ]:
%%script false
bu_time = df_done.groupby(['Month','BU'])['Patient Price, EUR'].sum().reset_index()
bu_time.columns = ['Month','BU','Revenue EUR']

# ---- Matplotlib --------
pivot_bu = bu_time.pivot_table(index='Month', columns='BU', values='Revenue EUR', fill_value=0)
fig, ax = plt.subplots()
pivot_bu.plot.area(ax=ax, color=[PALETTE.get(c,'#95A5A6') for c in pivot_bu.columns], alpha=0.8)
ax.set_title('BU Revenue Contribution over Time (EUR)')
ax.set_ylabel('Revenue EUR')
ax.tick_params(axis='x', rotation=45)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))
mpl_show(fig)

# ---- Plotly --------
pf = px.area(bu_time, x='Month', y='Revenue EUR', color='BU',
             color_discrete_map=PALETTE, labels={'Month':'','Revenue EUR':'Revenue EUR'})
plotly_capture(pf, '0.6 · BU Revenue Contribution over Time')

---
## Topic 1 — Revenue Analysis
### Chart 1.1 · BU Breakdown (README #1 — updated)

In [ ]:
bu_map = lambda x: 'BU Consultations' if x == 'BU Consultancy' else 'BU Others'
df_done['BU_display'] = df_done['BU'].apply(bu_map)

bu_rev   = df_done.groupby('BU_display')['Patient Price, EUR'].sum().reset_index()
bu_count = df_done.groupby('BU_display')['Record Id'].count().reset_index()
bu_rev.columns   = ['BU','Revenue EUR']
bu_count.columns = ['BU','Count']
PIE_COLORS = {'BU Consultations':'#2E86AB','BU Others':'#F18F01'}

# ---- Matplotlib --------
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, data, val_col, title in [
    (axes[0], bu_count, 'Count',       'Consultation Count by BU'),
    (axes[1], bu_rev,   'Revenue EUR', 'Revenue by BU (EUR)'),
]:
    wedge_colors = [PIE_COLORS.get(b,'#95A5A6') for b in data['BU']]
    wedges, texts, autotexts = ax.pie(
        data[val_col],
        labels=data['BU'],
        colors=wedge_colors,
        autopct='%1.1f%%',
        startangle=140,
        pctdistance=0.72,
        labeldistance=1.18,
        wedgeprops=dict(edgecolor='white', linewidth=3),
        textprops=dict(fontsize=11)
    )
    for at in autotexts:
        at.set_fontsize(10)
        at.set_fontweight('bold')
        at.set_color('white')
    ax.set_title(title, pad=20)
mpl_show(fig)

# ---- Plotly --------
pf = make_subplots(rows=1, cols=2, specs=[[{'type':'pie'},{'type':'pie'}]],
                   subplot_titles=['Consultation Count by BU','Revenue by BU (EUR)'])
pf.add_trace(go.Pie(labels=bu_count['BU'], values=bu_count['Count'],
                    marker_colors=[PIE_COLORS.get(b,'#95A5A6') for b in bu_count['BU']],
                    hole=0.35, textposition='inside',
                    insidetextorientation='radial', name='Count'), row=1, col=1)
pf.add_trace(go.Pie(labels=bu_rev['BU'], values=bu_rev['Revenue EUR'],
                    marker_colors=[PIE_COLORS.get(b,'#95A5A6') for b in bu_rev['BU']],
                    hole=0.35, textposition='inside',
                    insidetextorientation='radial', name='Revenue'), row=1, col=2)
pf.update_traces(textinfo='label+percent')
plotly_capture(pf, '1.1 · BU Breakdown — Count & Revenue (README #1)')

### Chart 1.2 · Price Realization Rate — Payment ÷ Tariff

In [ ]:
%%script false
# Only rows where both tariff > 0 and payment exists
real_df = df_done[(df_done['Patient Price, EUR'] > 0) &
                  (df_done['Patient payment , EUR'].notna())].copy()
real_df['Realization'] = (real_df['Patient payment , EUR'] / real_df['Patient Price, EUR']).clip(0, 2)
monthly_real = real_df.groupby(['Month','Service Type Group'])['Realization'].median().reset_index()
monthly_real = monthly_real[monthly_real['Service Type Group'].isin(CONSULTANCY)]

# ---- Matplotlib --------
fig, ax = plt.subplots()
for stype, color in zip(CONSULTANCY, ['#2E86AB','#A23B72']):
    sub = monthly_real[monthly_real['Service Type Group']==stype]
    ax.plot(sub['Month'], sub['Realization'], marker='o', linewidth=2,
            label=stype.replace(' Consultancy',''), color=color, markersize=4)
ax.axhline(1.0, color='grey', linestyle='--', linewidth=1, label='100% = full tariff collected')
ax.set_title('Price Realization Rate: Actual Payment ÷ Tariff (median)')
ax.set_ylabel('Realization Rate')
ax.tick_params(axis='x', rotation=45)
ax.legend()
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:.0%}'))
mpl_show(fig)

# ---- Plotly --------
pf = px.line(monthly_real, x='Month', y='Realization', color='Service Type Group',
             color_discrete_map=PALETTE, markers=True,
             labels={'Month':'','Realization':'Realization Rate','Service Type Group':'Service'})
pf.add_hline(y=1.0, line_dash='dash', line_color='grey', annotation_text='100%')
pf.update_yaxes(tickformat='.0%')
plotly_capture(pf, '1.2 · Price Realization Rate (Payment ÷ Tariff)')

### Chart 1.3 · Dynamic Service Type Share by Quantity (README #2 — updated)

In [ ]:
monthly = (df_consult.groupby(['Month','Service Type Group'])
           .size().unstack(fill_value=0))
monthly_total = monthly.sum(axis=1)
pct = monthly.div(monthly_total, axis=0) * 100

# Find global max and min of total volume
max_month = monthly_total.idxmax()
min_month = monthly_total.idxmin()
max_val   = monthly_total[max_month]
min_val   = monthly_total[min_month]
max_idx   = list(monthly_total.index).index(max_month)
min_idx   = list(monthly_total.index).index(min_month)

# ---- Matplotlib --------
fig, ax = plt.subplots(figsize=(13,5))
pct.plot.area(ax=ax, color=[PALETTE.get(c,'#999') for c in pct.columns], alpha=0.8)
ax.set_title('Service Type Share of Total Consultations (100% area)')
ax.set_ylabel('Share %')
ax.set_ylim(0, 100)
ax.tick_params(axis='x', rotation=45)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:.0f}%'))

# Extremum markers on top edge (at 100% line)
ax.annotate(f'Peak\n{max_val:,}', xy=(max_idx, 98),
            xytext=(max_idx, 108), textcoords='data',
            ha='center', fontsize=8, color='#27AE60',
            arrowprops=dict(arrowstyle='->', color='#27AE60', lw=1.2))
ax.annotate(f'Low\n{min_val:,}', xy=(min_idx, 98),
            xytext=(min_idx, 108), textcoords='data',
            ha='center', fontsize=8, color='#E74C3C',
            arrowprops=dict(arrowstyle='->', color='#E74C3C', lw=1.2))
ax.set_ylim(0, 115)
mpl_show(fig)

# ---- Plotly --------
m = (df_consult.groupby(['Month','Service Type Group']).size().reset_index(name='Count'))
totals = m.groupby('Month')['Count'].sum().reset_index(name='Total')
m = m.merge(totals, on='Month')
m['Share'] = m['Count'] / m['Total'] * 100

pf = px.area(m, x='Month', y='Share', color='Service Type Group',
             color_discrete_map=PALETTE, groupnorm='percent',
             labels={'Month':'','Share':'Share %','Service Type Group':'Service'})
# Use add_shape + add_annotation (add_vline fails on string x-axes)
for xval, color, label, xanchor in [
    (max_month, '#27AE60', f'Peak {max_val:,}', 'left'),
    (min_month, '#E74C3C', f'Low {min_val:,}',  'right'),
]:
    pf.add_shape(type='line', x0=xval, x1=xval, y0=0, y1=1,
                 xref='x', yref='paper',
                 line=dict(color=color, width=1.5, dash='dot'))
    pf.add_annotation(x=xval, y=1.05, text=label,
                      xref='x', yref='paper',
                      showarrow=False, xanchor=xanchor,
                      font=dict(color=color, size=10))
plotly_capture(pf, '1.3 · Dynamic Service Type Share by Quantity')

---
## Topic 2 — Pricing Analysis
### Chart 2.1 · Chat vs Video Prices — Boxplot (README #8 — outliers removed)

In [ ]:

def annotate_boxplot(ax, data_list, labels, x_offset=0.06):
    """Annotate boxplot with median, Q1, Q3, whisker min/max."""
    for i, (data, label) in enumerate(zip(data_list, labels), start=1):
        q1  = data.quantile(0.25)
        med = data.quantile(0.50)
        q3  = data.quantile(0.75)
        iqr = q3 - q1
        wlo = data[data >= q1 - 1.5*iqr].min()
        whi = data[data <= q3 + 1.5*iqr].max()
        for val, label_txt in [(whi, f'Max\nEUR {whi:.0f}'),
                               (q3,  f'Q3 EUR {q3:.0f}'),
                               (med, f'Med EUR {med:.0f}'),
                               (q1,  f'Q1 EUR {q1:.0f}'),
                               (wlo, f'Min\nEUR {wlo:.0f}')]:
            ax.text(i + x_offset, val, label_txt,
                    va='center', fontsize=7, color='#2C3E50')

price_data = [df_consult[df_consult['Service Type Group']==s]['Patient Price, EUR']
                .dropna().clip(upper=P99_PRICE) for s in CONSULTANCY]

# ---- Matplotlib --------
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
bp = axes[0].boxplot(price_data, labels=['Video','Chat'], patch_artist=True,
                     showfliers=False, medianprops=dict(color='white', linewidth=2))
for patch, color in zip(bp['boxes'], ['#2E86AB','#A23B72']):
    patch.set_facecolor(color)
annotate_boxplot(axes[0], price_data, CONSULTANCY)
axes[0].set_title(f'Price Boxplot — clipped at p99 (EUR {P99_PRICE:.0f})')
axes[0].set_ylabel('Patient Price EUR')

for s, color in zip(CONSULTANCY, ['#2E86AB','#A23B72']):
    (df_consult[df_consult['Service Type Group']==s]['Patient Price, EUR']
     .dropna().clip(upper=P99_PRICE)
     .plot.hist(bins=40, ax=axes[1], alpha=0.6, color=color,
                label=s.replace(' Consultancy',''), edgecolor='white'))
axes[1].set_title(f'Price Histogram (capped at EUR {P99_PRICE:.0f})')
axes[1].set_xlabel('Consultation Price (EUR)')
axes[1].legend()
mpl_show(fig)

# ---- Plotly --------
df_plot = df_consult[['Service Type Group','Patient Price, EUR']].copy()
df_plot['Patient Price, EUR'] = df_plot['Patient Price, EUR'].clip(upper=P99_PRICE)
pf = px.box(df_plot, x='Service Type Group', y='Patient Price, EUR',
            color='Service Type Group', color_discrete_map=PALETTE, points=False,
            labels={'Service Type Group':'Service',
                    'Patient Price, EUR':f'Price EUR (capped p99=EUR {P99_PRICE:.0f})'})
plotly_capture(pf, '2.1 · Chat vs Video Price Boxplot (README #8)')

### Chart 2.2 · Monthly Median Tariff Price Trend

In [ ]:
price_trend = (df_consult[df_consult['Patient Price, EUR'] > 0]
               .groupby(['Month','Service Type Group'])['Patient Price, EUR']
               .median().reset_index())

# ---- Matplotlib --------
fig, ax = plt.subplots(figsize=(13,5))
for stype, color in zip(CONSULTANCY, ['#2E86AB','#A23B72']):
    sub = price_trend[price_trend['Service Type Group']==stype].sort_values('Month')
    ax.plot(sub['Month'], sub['Patient Price, EUR'], marker='o', linewidth=2,
            label=stype.replace(' Consultancy',''), color=color, markersize=5)

ax.set_title('Monthly Median Consultation Price — Video vs Chat (EUR)')
ax.set_ylabel('Median Price EUR')
ax.tick_params(axis='x', rotation=45)
ax.legend()
mpl_show(fig)

# ---- Plotly --------
pf = px.line(price_trend, x='Month', y='Patient Price, EUR', color='Service Type Group',
             color_discrete_map=PALETTE, markers=True,
             labels={'Month':'','Patient Price, EUR':'Median Price EUR',
                     'Service Type Group':'Service'})
plotly_capture(pf, '2.2 · Monthly Median Price Trend')

### Chart 2.3 · Effective Price per Patient Currency

In [ ]:
%%script false
price_ccy = (df_done[df_done['Patient Price, EUR'] > 0]
             .groupby('Patient Payment Currency')
             .agg(median_price=('Patient Price, EUR','median'), n=('Record Id','count'))
             .reset_index())
price_ccy = price_ccy[price_ccy['n'] >= 10]

# ---- Matplotlib --------
fig, ax = plt.subplots(figsize=(8,4))
bars = ax.bar(price_ccy['Patient Payment Currency'], price_ccy['median_price'],
              color=[PALETTE.get(c,'#95A5A6') for c in price_ccy['Patient Payment Currency']],
              edgecolor='white')
for bar, n in zip(bars, price_ccy['n']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'n={n:,}', ha='center', va='bottom', fontsize=9)
ax.set_title('Median EUR-normalised Tariff Price by Patient Payment Currency')
ax.set_ylabel('Median Price EUR')
mpl_show(fig)

# ---- Plotly --------
pf = px.bar(price_ccy, x='Patient Payment Currency', y='median_price',
            color='Patient Payment Currency', color_discrete_map=PALETTE,
            text=price_ccy.apply(lambda r: f'EUR{r.median_price:.0f} (n={r.n:,})',axis=1),
            labels={'median_price':'Median Tariff EUR','Patient Payment Currency':'Currency'})
pf.update_traces(textposition='outside')
plotly_capture(pf, '2.3 · Effective Price per Patient Currency')

---
## Topic 3 — Volume & Cancellation
### Chart 3.1 · Cancellation Trend over Time (README #6 — updated)

In [ ]:
ms = (df_consult_all.groupby(['Month','StageGroup']).size().unstack(fill_value=0))
show_s = [c for c in ['Closed','Cancelled','Stage others'] if c in ms.columns]

# ---- Matplotlib --------
fig, ax = plt.subplots()
for col in show_s:
    ax.plot(ms.index, ms[col], marker='o', linewidth=2,
            label=col, color=PALETTE.get(col,'#999'), markersize=4)
ax.set_title('Monthly Stage Trend — Consultancy BU')
ax.set_ylabel('Records'); ax.tick_params(axis='x', rotation=45); ax.legend()
mpl_show(fig)

# ---- Plotly --------
tr = df_consult_all.groupby(['Month','StageGroup']).size().reset_index(name='Count')
tr = tr[tr['StageGroup'].isin(show_s)]
pf = px.line(tr, x='Month', y='Count', color='StageGroup', markers=True,
             color_discrete_map=PALETTE,
             labels={'Month':'','Count':'Records','StageGroup':'Stage'})
plotly_capture(pf, '3.1 · Cancellation Trend over Time (README #6)')

### Chart 3.4 · Revenue Foregone from Cancellations

In [ ]:
cancelled = df_all[(df_all['StageGroup']=='Cancelled') &
                    (df_all['Patient Price, EUR'].notna())].copy()
# Clip price at p99 before computing lost revenue
cancelled['Price_clipped'] = cancelled['Patient Price, EUR'].clip(upper=P99_PRICE)
lost = cancelled.groupby('Month')['Price_clipped'].sum().reset_index(name='Lost Revenue EUR')

# ---- Matplotlib --------
fig, ax = plt.subplots()
ax.fill_between(lost['Month'], lost['Lost Revenue EUR'], alpha=0.25, color='#E74C3C')
ax.plot(lost['Month'], lost['Lost Revenue EUR'], color='#E74C3C', linewidth=2, marker='o', markersize=4)
ax.set_title(f'Monthly Revenue Foregone from Cancellations (EUR, clipped at p99=EUR {P99_PRICE:.0f})')
ax.set_ylabel('Lost Revenue EUR')
ax.tick_params(axis='x', rotation=45)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))
mpl_show(fig)

# ---- Plotly --------
pf = px.area(lost, x='Month', y='Lost Revenue EUR', markers=True,
             color_discrete_sequence=['#E74C3C'],
             labels={'Month':'','Lost Revenue EUR':'Lost Revenue EUR'})
plotly_capture(pf, '3.4 · Monthly Revenue Foregone from Cancellations')

### Chart 3.3 · No-show vs Cancellation Monthly

In [ ]:
%%script false
nshow = df_consult_all.groupby(['Month','StageGroup']).size().unstack(fill_value=0)
ns_cols = [c for c in ['Cancelled','Stage others'] if c in nshow.columns]

# ---- Matplotlib --------
fig, ax = plt.subplots()
width = 0.35
x = np.arange(len(nshow.index))
if 'Cancelled' in nshow.columns:
    ax.bar(x - width/2, nshow['Cancelled'], width, color='#E74C3C', alpha=0.8, label='Cancelled')
if 'Stage others' in nshow.columns:
    ax.bar(x + width/2, nshow['Stage others'], width, color='#F39C12', alpha=0.8, label='No-show / Other')
ax.set_xticks(x); ax.set_xticklabels(nshow.index, rotation=45, ha='right')
ax.set_title('Monthly Cancelled vs No-show/Other — Consultancy BU')
ax.set_ylabel('Records'); ax.legend()
mpl_show(fig)

# ---- Plotly --------
tr2 = df_consult_all[df_consult_all['StageGroup'].isin(ns_cols)].groupby(
    ['Month','StageGroup']).size().reset_index(name='Count')
pf = px.bar(tr2, x='Month', y='Count', color='StageGroup', barmode='group',
            color_discrete_map=PALETTE,
            labels={'Month':'','Count':'Records','StageGroup':'Stage'})
plotly_capture(pf, '3.3 · No-show vs Cancellation Monthly')

### Chart 3.2 · Cancellation Rate by Doctor (min 5 consultations)

In [ ]:
doc_stats = df_all.groupby('Doctor.id').agg(
    Total=('Record Id','count'),
    Cancelled=('StageGroup', lambda x: (x=='Cancelled').sum())
).reset_index()
doc_stats = doc_stats[doc_stats['Total'] >= 5].copy()
doc_stats['CancelRate'] = doc_stats['Cancelled'] / doc_stats['Total'] * 100
top_cancel = doc_stats.nlargest(20, 'CancelRate').sort_values('CancelRate')

# ---- Matplotlib --------
fig, ax = plt.subplots(figsize=(11,6))
ax.barh(top_cancel['Doctor.id'].astype(str), top_cancel['CancelRate'],
        color='#E74C3C', edgecolor='white', alpha=0.8)
ax.set_title('Top 20 Doctors by Cancellation Rate (min 5 consultations)')
ax.set_xlabel('Cancellation Rate %')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:.0f}%'))
mpl_show(fig)

# ---- Plotly --------
top_cancel['Doctor'] = top_cancel['Doctor.id'].astype(str)
pf = px.bar(top_cancel, x='CancelRate', y='Doctor', orientation='h',
            color='CancelRate', color_continuous_scale=['#F39C12','#E74C3C'],
            hover_data=['Total','Cancelled'],
            labels={'CancelRate':'Cancellation Rate %','Doctor':''})
plotly_capture(pf, '3.2 · Top 20 Doctors by Cancellation Rate')

---
## Topic 4 — Cost Analysis
### Chart 4.1 · Doctor Cost by Service Type — Boxplot

In [ ]:

def annotate_boxplot(ax, data_list, labels, x_offset=0.06):
    """Annotate boxplot with median, Q1, Q3, whisker min/max."""
    for i, (data, label) in enumerate(zip(data_list, labels), start=1):
        q1  = data.quantile(0.25)
        med = data.quantile(0.50)
        q3  = data.quantile(0.75)
        iqr = q3 - q1
        wlo = data[data >= q1 - 1.5*iqr].min()
        whi = data[data <= q3 + 1.5*iqr].max()
        for val, label_txt in [(whi, f'Max\nEUR {whi:.0f}'),
                               (q3,  f'Q3 EUR {q3:.0f}'),
                               (med, f'Med EUR {med:.0f}'),
                               (q1,  f'Q1 EUR {q1:.0f}'),
                               (wlo, f'Min\nEUR {wlo:.0f}')]:
            ax.text(i + x_offset, val, label_txt,
                    va='center', fontsize=7, color='#2C3E50')

cost_data = [df_consult[df_consult['Service Type Group']==s]['Doctor cost, EUR']
                .dropna().clip(upper=P99_COST) for s in CONSULTANCY]

# ---- Matplotlib --------
fig, ax = plt.subplots(figsize=(8, 5))
bp = ax.boxplot(cost_data, labels=['Video','Chat'], patch_artist=True,
                showfliers=False, medianprops=dict(color='white', linewidth=2))
for patch, color in zip(bp['boxes'], ['#2E86AB','#A23B72']):
    patch.set_facecolor(color)
annotate_boxplot(ax, cost_data, CONSULTANCY)
ax.set_title(f'Doctor Cost Distribution — clipped at p99 (EUR {P99_COST:.0f})')
ax.set_ylabel('Doctor Cost EUR')
mpl_show(fig)

# ---- Plotly --------
df_cplot = df_consult[['Service Type Group','Doctor cost, EUR']].copy()
df_cplot['Doctor cost, EUR'] = df_cplot['Doctor cost, EUR'].clip(upper=P99_COST)
pf = px.box(df_cplot, x='Service Type Group', y='Doctor cost, EUR',
            color='Service Type Group', color_discrete_map=PALETTE, points=False,
            labels={'Service Type Group':'Service',
                    'Doctor cost, EUR':f'Cost EUR (capped p99=EUR {P99_COST:.0f})'})
plotly_capture(pf, '4.1 · Doctor Cost by Service Type')

### Chart 4.2 · Cost Currency Mix

In [ ]:
%%script false
cost_ccy = df_done.groupby('Doctor Cost Currency')['Doctor cost, EUR'].sum().reset_index()
cost_ccy.columns = ['Currency','Cost EUR']
cost_ccy = cost_ccy[cost_ccy['Cost EUR'] > 0].sort_values('Cost EUR', ascending=False)

# ---- Matplotlib --------
fig, axes = plt.subplots(1, 2, figsize=(12,4))
cost_ccy.plot.bar(ax=axes[0], x='Currency', y='Cost EUR',
                  color=[PALETTE.get(c,'#95A5A6') for c in cost_ccy['Currency']],
                  edgecolor='white', legend=False, rot=0)
axes[0].set_title('Doctor Cost by Currency (EUR-norm)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))
axes[1].pie(cost_ccy['Cost EUR'], labels=cost_ccy['Currency'],
            colors=[PALETTE.get(c,'#95A5A6') for c in cost_ccy['Currency']],
            autopct='%1.1f%%', startangle=90)
axes[1].set_title('Cost Currency Share')
mpl_show(fig)

# ---- Plotly --------
pf = px.bar(cost_ccy, x='Currency', y='Cost EUR', color='Currency',
            color_discrete_map=PALETTE,
            text=cost_ccy['Cost EUR'].apply(lambda x: f'EUR{x:,.0f}'),
            labels={'Cost EUR':'EUR-norm cost'})
pf.update_traces(textposition='outside')
plotly_capture(pf, '4.2 · Doctor Cost Currency Mix')

### Chart 4.3 · Monthly Median Doctor Cost Trend

In [ ]:
%%script false
cost_trend = (df_consult[df_consult['Doctor cost, EUR'] > 0]
               .groupby(['Month','Service Type Group'])['Doctor cost, EUR']
               .median().reset_index())

# ---- Matplotlib --------
fig, ax = plt.subplots()
for stype, color in zip(CONSULTANCY, ['#2E86AB','#A23B72']):
    sub = cost_trend[cost_trend['Service Type Group']==stype]
    ax.plot(sub['Month'], sub['Doctor cost, EUR'], marker='o', linewidth=2,
            label=stype.replace(' Consultancy',''), color=color, markersize=4)
ax.set_title('Monthly Median Doctor Cost — Video vs Chat (EUR)')
ax.set_ylabel('Median Doctor Cost EUR')
ax.tick_params(axis='x', rotation=45)
ax.legend()
mpl_show(fig)

# ---- Plotly --------
pf = px.line(cost_trend, x='Month', y='Doctor cost, EUR', color='Service Type Group',
             color_discrete_map=PALETTE, markers=True,
             labels={'Month':'','Doctor cost, EUR':'Median Cost EUR','Service Type Group':'Service'})
plotly_capture(pf, '4.3 · Monthly Median Doctor Cost Trend')

### Chart 4.4 · Cost Pareto — Doctors → 80% of Total Cost

In [ ]:
%%script false
doc_cost = df_done.groupby('Doctor.id')['Doctor cost, EUR'].sum().sort_values(ascending=False)
cum_cost = doc_cost.cumsum() / doc_cost.sum() * 100
nc80 = int((cum_cost >= 80).argmax()) + 1
cpct80 = nc80 / len(doc_cost) * 100

# ---- Matplotlib --------
fig, ax1 = plt.subplots()
ax1.bar(range(len(doc_cost)), doc_cost.values, color='#E74C3C', alpha=0.65, width=1)
ax1.set_ylabel('Doctor Cost EUR', color='#E74C3C')
ax2 = ax1.twinx()
ax2.plot(range(len(doc_cost)), cum_cost.values, color='#2C3E50', linewidth=2)
ax2.axhline(80, color='grey', linestyle='--'); ax2.axvline(nc80, color='grey', linestyle='--')
ax2.set_ylabel('Cumulative %', color='#2C3E50'); ax2.set_ylim(0,105)
ax1.set_title(f'Pareto — Doctor Costs: top {cpct80:.0f}% of doctors → 80% of total cost')
mpl_show(fig)

# ---- Plotly --------
dfc = doc_cost.reset_index(); dfc.columns=['Doctor','Cost']
dfc['CumPct'] = doc_cost.cumsum().values/doc_cost.sum()*100
dfc['Rank'] = range(1,len(dfc)+1)
pf = make_subplots(specs=[[{'secondary_y':True}]])
pf.add_trace(go.Bar(x=dfc['Rank'],y=dfc['Cost'],marker_color='#E74C3C',opacity=0.65,name='Cost EUR'),secondary_y=False)
pf.add_trace(go.Scatter(x=dfc['Rank'],y=dfc['CumPct'],line=dict(color='#2C3E50',width=2),name='Cumulative %'),secondary_y=True)
pf.add_hline(y=80,line_dash='dash',line_color='grey',secondary_y=True)
pf.add_vline(x=nc80,line_dash='dash',line_color='grey')
pf.update_yaxes(title_text='Cost EUR',secondary_y=False)
pf.update_yaxes(title_text='Cumulative %',range=[0,105],secondary_y=True)
pf.update_xaxes(title_text=f'Doctors ranked — top {cpct80:.0f}% → 80% of cost')
plotly_capture(pf, '4.4 · Cost Pareto — Doctors')

### Chart 4.5 · Price / Cost / Gross Profit Distributions (README #9 — clipped)

In [ ]:
# 4.5 — All records: Price / Cost / Unit Gross Profit histograms
pairs_all = [
    (df_done['Patient Price, EUR'].dropna().clip(upper=P99_PRICE),
     f'Patient Price EUR (p99=EUR {P99_PRICE:.0f})', '#2E86AB'),
    (df_done['Doctor cost, EUR'].dropna().clip(upper=P99_COST),
     f'Doctor Cost EUR (p99=EUR {P99_COST:.0f})', '#E74C3C'),
    (df_done['Unit Gross profit'].dropna().clip(P01_GP, P99_GP),
     'Unit Gross Profit EUR (p1–p99)', '#27AE60'),
]

# ---- Matplotlib --------
fig, axes = plt.subplots(1, 3, figsize=(15,4))
fig.suptitle('Price / Cost / Unit Gross Profit — All Records', fontweight='bold')
for ax, (series, title, color) in zip(axes, pairs_all):
    ax.hist(series, bins=50, color=color, edgecolor='white', alpha=0.85)
    med = series.median()
    ax.axvline(med, color='black', linestyle='--', linewidth=1.2)
    ax.text(med, ax.get_ylim()[1]*0.9, f'  Median\n  {med:.0f}', fontsize=8)
    ax.set_title(title)
mpl_show(fig)

# ---- Plotly --------
pf = make_subplots(rows=1, cols=3, subplot_titles=[p[1] for p in pairs_all])
for col,(series,title,color) in enumerate(pairs_all, start=1):
    med = series.median()
    pf.add_trace(go.Histogram(x=series, nbinsx=50, marker_color=color,
                              opacity=0.85, name=title), row=1, col=col)
    pf.add_vline(x=med, line_dash='dash', line_color='black',
                 annotation_text=f'Median {med:.0f}', row=1, col=col)
plotly_capture(pf, '4.5 · Price / Cost / Unit GP Distributions — All Records (README #9)')

In [ ]:
# 4.5 Overlay — Video vs Chat: Price / Cost / Unit GP (overlaid histograms)
col_pairs = [
    ('Patient Price, EUR',   P99_PRICE, P99_PRICE, 'Patient Price EUR'),
    ('Doctor cost, EUR',     P99_COST,  P99_COST,  'Doctor Cost EUR'),
    ('Unit Gross profit',    P01_GP,    P99_GP,    'Unit Gross Profit EUR'),
]

# ---- Matplotlib --------
fig, axes = plt.subplots(1, 3, figsize=(15,4))
fig.suptitle('Price / Cost / Unit GP — Video vs Chat Overlay', fontweight='bold')
for ax, (col, lo, hi, title) in zip(axes, col_pairs):
    for stype, color in zip(CONSULTANCY, ['#2E86AB','#A23B72']):
        series = (df_consult[df_consult['Service Type Group']==stype][col]
                  .dropna().clip(lo if col=='Unit Gross profit' else 0, hi))
        ax.hist(series, bins=40, color=color, alpha=0.55, edgecolor='white',
                label=stype.replace(' Consultancy',''))
        ax.axvline(series.median(), color=color, linestyle='--', linewidth=1.2)
    ax.set_title(title)
    ax.legend(fontsize=8)
mpl_show(fig)

# ---- Plotly --------
pf = make_subplots(rows=1, cols=3,
                   subplot_titles=[c[3] for c in col_pairs])
for col_idx, (col, lo, hi, title) in enumerate(col_pairs, start=1):
    for stype, color in zip(CONSULTANCY, ['#2E86AB','#A23B72']):
        series = (df_consult[df_consult['Service Type Group']==stype][col]
                  .dropna().clip(lo if col=='Unit Gross profit' else 0, hi))
        pf.add_trace(go.Histogram(x=series, nbinsx=40,
                                   marker_color=color, opacity=0.55,
                                   name=stype.replace(' Consultancy',''),
                                   showlegend=(col_idx==1)),
                     row=1, col=col_idx)
pf.update_layout(barmode='overlay')
plotly_capture(pf, '4.5 Overlay · Video vs Chat: Price / Cost / Unit GP')

---
## Topic 5 — Margin Analysis
### Chart 5.1 · Gross Profit by Service Type

In [ ]:
gp_svc = df_consult.groupby('Service Type Group').agg(
    MedianGP=('Gross profit','median'),
    TotalGP=('Gross profit','sum'),
    Count=('Record Id','count')
).reset_index()

# ---- Matplotlib --------
fig, axes = plt.subplots(1, 3, figsize=(16,5))

# Left: median GP per consultation
bars0 = axes[0].bar(gp_svc['Service Type Group'], gp_svc['MedianGP'],
                    color=[PALETTE.get(s,'#95A5A6') for s in gp_svc['Service Type Group']],
                    edgecolor='white')
axes[0].axhline(0, color='grey', linestyle='--')
axes[0].set_title('Median GP per Consultation (EUR)')
axes[0].set_ylabel('Median Gross Profit EUR')
axes[0].tick_params(axis='x', rotation=15)
for bar in bars0:
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                 f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=9)

# Middle: total GP
bars1 = axes[1].bar(gp_svc['Service Type Group'], gp_svc['TotalGP'],
                    color=[PALETTE.get(s,'#95A5A6') for s in gp_svc['Service Type Group']],
                    edgecolor='white')
axes[1].axhline(0, color='grey', linestyle='--')
axes[1].set_title('Total GP (EUR)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))
axes[1].tick_params(axis='x', rotation=15)
for bar in bars1:
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+50,
                 f'{bar.get_height():,.0f}', ha='center', va='bottom', fontsize=9)

# Right: consultation count
bars2 = axes[2].bar(gp_svc['Service Type Group'], gp_svc['Count'],
                    color=[PALETTE.get(s,'#95A5A6') for s in gp_svc['Service Type Group']],
                    edgecolor='white')
axes[2].set_title('Consultation Count')
axes[2].set_ylabel('Count')
axes[2].tick_params(axis='x', rotation=15)
for bar in bars2:
    axes[2].text(bar.get_x()+bar.get_width()/2, bar.get_height()+5,
                 f'{bar.get_height():,.0f}', ha='center', va='bottom', fontsize=9)

mpl_show(fig)

# ---- Plotly --------
pf = make_subplots(rows=1, cols=3,
                   subplot_titles=['Median GP per Consultation','Total GP (EUR)','Consultation Count'])
for col_i, (y_col, name) in enumerate([('MedianGP','Median GP'),
                                        ('TotalGP','Total GP'),
                                        ('Count','Count')], start=1):
    pf.add_trace(go.Bar(
        x=gp_svc['Service Type Group'], y=gp_svc[y_col],
        marker_color=[PALETTE.get(s,'#95A5A6') for s in gp_svc['Service Type Group']],
        text=gp_svc[y_col].apply(lambda v: f'{v:,.1f}' if col_i<3 else f'{v:,}'),
        textposition='outside', name=name, showlegend=False
    ), row=1, col=col_i)
    if col_i < 3:
        pf.add_hline(y=0, line_dash='dash', line_color='grey', row=1, col=col_i)
plotly_capture(pf, '5.1 · Gross Profit by Service Type + Count')

### Chart 5.2 · Monthly Gross Profit Trend

In [ ]:
%%script false
monthly_gp = df_done.groupby('Month')['Gross profit'].sum().reset_index()

# ---- Matplotlib --------
fig, ax = plt.subplots()
colors = ['#27AE60' if v >= 0 else '#E74C3C' for v in monthly_gp['Gross profit']]
ax.bar(monthly_gp['Month'], monthly_gp['Gross profit'], color=colors, alpha=0.75)
ax.axhline(0, color='grey', linestyle='--')
ax.plot(monthly_gp['Month'], monthly_gp['Gross profit'].rolling(3, center=True).mean(),
        color='#2C3E50', linewidth=2, label='3-month rolling avg')
ax.set_title('Monthly Total Gross Profit (EUR)')
ax.set_ylabel('Gross Profit EUR')
ax.tick_params(axis='x', rotation=45)
ax.legend()
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))
mpl_show(fig)

# ---- Plotly --------
monthly_gp['color'] = monthly_gp['Gross profit'].apply(lambda x: '#27AE60' if x>=0 else '#E74C3C')
monthly_gp['Rolling'] = monthly_gp['Gross profit'].rolling(3,center=True).mean()
pf = go.Figure()
pf.add_trace(go.Bar(x=monthly_gp['Month'],y=monthly_gp['Gross profit'],
                    marker_color=monthly_gp['color'],opacity=0.75,name='Monthly GP'))
pf.add_trace(go.Scatter(x=monthly_gp['Month'],y=monthly_gp['Rolling'],
                        line=dict(color='#2C3E50',width=2),name='3M rolling avg'))
pf.add_hline(y=0,line_dash='dash',line_color='grey')
plotly_capture(pf,'5.2 · Monthly Gross Profit Trend')

### Chart 5.3 · Top 20 Doctors by Gross Profit

In [ ]:
%%script false
doc_gp = df_done.groupby('Doctor.id')['Gross profit'].sum().nlargest(20).sort_values()

# ---- Matplotlib --------
fig, ax = plt.subplots(figsize=(11,6))
colors = ['#27AE60' if v >= 0 else '#E74C3C' for v in doc_gp.values]
ax.barh(doc_gp.index.astype(str), doc_gp.values, color=colors, edgecolor='white')
ax.axvline(0, color='grey', linestyle='--')
ax.set_title('Top 20 Doctors by Total Gross Profit (EUR)')
ax.set_xlabel('Total Gross Profit EUR')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))
mpl_show(fig)

# ---- Plotly --------
dg = doc_gp.reset_index(); dg.columns=['Doctor','GP']
dg['Doctor'] = dg['Doctor'].astype(str)
dg['Color'] = dg['GP'].apply(lambda x:'#27AE60' if x>=0 else '#E74C3C')
pf = px.bar(dg,x='GP',y='Doctor',orientation='h',color='Color',
            color_discrete_map={'#27AE60':'#27AE60','#E74C3C':'#E74C3C'},
            labels={'GP':'Total Gross Profit EUR','Doctor':''})
pf.add_vline(x=0,line_dash='dash',line_color='grey')
plotly_capture(pf,'5.3 · Top 20 Doctors by Gross Profit')

### Chart 5.4 · Median Gross Profit by Doctor Specialty Group

In [ ]:
%%script false
if 'Doctor Specialty Group' in df_done.columns:
    spec_gp = (df_done.groupby('Doctor Specialty Group')['Gross profit']
               .median().dropna().sort_values(ascending=True).tail(15))
    
    # ---- Matplotlib --------
    fig, ax = plt.subplots(figsize=(11,5))
    colors = ['#27AE60' if v >= 0 else '#E74C3C' for v in spec_gp.values]
    spec_gp.plot.barh(ax=ax, color=colors, edgecolor='white')
    ax.axvline(0, color='grey', linestyle='--')
    ax.set_title('Median Gross Profit per Consultation by Specialty Group (EUR)')
    ax.set_xlabel('Median GP EUR')
    mpl_show(fig)
    
    # ---- Plotly --------
    sg = spec_gp.reset_index(); sg.columns=['Specialty','MedianGP']
    sg['Color'] = sg['MedianGP'].apply(lambda x:'#27AE60' if x>=0 else '#E74C3C')
    pf = px.bar(sg,x='MedianGP',y='Specialty',orientation='h',color='Color',
                color_discrete_map={'#27AE60':'#27AE60','#E74C3C':'#E74C3C'},
                labels={'MedianGP':'Median GP EUR','Specialty':''})
    pf.add_vline(x=0,line_dash='dash',line_color='grey')
    plotly_capture(pf,'5.4 · Median GP by Doctor Specialty Group')
else:
    print('Specialty Group column not found — check specialty join in setup cell')

### Chart 5.5 · Consultation Quantity vs Gross Profit (README #12 — clipped)

In [ ]:
doc_grp = df_done.groupby('Doctor.id').agg(
    Quantity=('Record Id','count'),
    UnitGP=('Unit Gross profit','mean'),
    Revenue=('Patient Price, EUR','sum')
).dropna().reset_index()

P01_UGP = df_done['Unit Gross profit'].quantile(0.01)
P99_UGP = df_done['Unit Gross profit'].quantile(0.99)

# ---- Matplotlib --------
fig, ax = plt.subplots()
ax.scatter(doc_grp['Quantity'], doc_grp['UnitGP'],
           alpha=0.5, color='#2E86AB', edgecolors='white', s=50)
ax.axhline(0, color='grey', linestyle='--', linewidth=0.8)
ax.set_title('Doctor: Consultation Quantity vs Avg Unit Gross Profit')
ax.set_xlabel('Number of Consultations')
ax.set_ylabel('Avg Unit Gross Profit (EUR)')
ax.set_ylim(P01_UGP-5, P99_UGP+5)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))
mpl_show(fig)

# ---- Plotly --------
pf = px.scatter(doc_grp, x='Quantity', y='UnitGP', size='Revenue',
                color='Revenue', color_continuous_scale='Blues',
                hover_data=['Doctor.id','Revenue'],
                range_y=[P01_UGP-5, P99_UGP+5],
                labels={'Quantity':'Consultations','UnitGP':'Avg Unit GP EUR'})
pf.add_hline(y=0, line_dash='dash', line_color='grey')
plotly_capture(pf, '5.5 · Quantity vs Unit Gross Profit (README #12)')

---
## Topic 6 — FX Impact Analysis
### Chart 6.2 · Native vs EUR-Normalised Revenue by Currency

In [ ]:
%%script false
rev_native = df_done.groupby('Patient Payment Currency').agg(
    NativeAmt=('Patient Payment Amount','sum'),
    EURnorm=('Patient payment , EUR','sum')
).dropna().reset_index()
rev_native = rev_native[rev_native['EURnorm'] > 0]
rev_native_m = rev_native.melt(id_vars='Patient Payment Currency',
                                value_vars=['NativeAmt','EURnorm'],
                                var_name='Type', value_name='Amount')

# ---- Matplotlib --------
fig, ax = plt.subplots(figsize=(10,4))
x = np.arange(len(rev_native))
w = 0.35
ax.bar(x - w/2, rev_native['NativeAmt'], w, color='#95A5A6', edgecolor='white', label='Native currency amount')
ax.bar(x + w/2, rev_native['EURnorm'],   w, color='#2E86AB', edgecolor='white', label='EUR-normalised')
ax.set_xticks(x); ax.set_xticklabels(rev_native['Patient Payment Currency'])
ax.set_title('Patient Revenue: Native Amount vs EUR-normalised')
ax.set_ylabel('Amount')
ax.legend()
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))
mpl_show(fig)

# ---- Plotly --------
pf = px.bar(rev_native_m, x='Patient Payment Currency', y='Amount',
            color='Type', barmode='group',
            color_discrete_map={'NativeAmt':'#95A5A6','EURnorm':'#2E86AB'},
            labels={'Patient Payment Currency':'Currency','Type':'Metric'})
plotly_capture(pf,'6.2 · Native vs EUR-normalised Revenue by Currency')

### Chart 6.3 · RUB/EUR FX Rate vs Monthly Gross Profit

In [ ]:
%%script false
fx['Date'] = pd.to_datetime(fx['Date'])
fx_rub = fx[fx['Currency']=='RUB'].copy()
fx_rub['Month'] = fx_rub['Date'].dt.to_period('M').astype(str)
fx_monthly = fx_rub.groupby('Month')['FX_Rate_EUR'].mean().reset_index(name='RUB_per_EUR')
gp_monthly = df_done.groupby('Month')['Gross profit'].sum().reset_index()
fx_gp = fx_monthly.merge(gp_monthly, on='Month')

# ---- Matplotlib --------
fig, ax1 = plt.subplots()
ax1.plot(fx_gp['Month'], fx_gp['RUB_per_EUR'], color='#F18F01', linewidth=2, marker='o', markersize=3, label='RUB/EUR rate')
ax1.set_ylabel('RUB per EUR', color='#F18F01')
ax1.tick_params(axis='x', rotation=45)
ax2 = ax1.twinx()
ax2.bar(fx_gp['Month'], fx_gp['Gross profit'], color='#27AE60', alpha=0.4, label='Monthly GP')
ax2.axhline(0, color='grey', linestyle='--')
ax2.set_ylabel('Gross Profit EUR', color='#27AE60')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))
ax1.set_title('RUB/EUR FX Rate vs Monthly Gross Profit')
mpl_show(fig)

# ---- Plotly --------
pf = make_subplots(specs=[[{'secondary_y':True}]])
pf.add_trace(go.Scatter(x=fx_gp['Month'],y=fx_gp['RUB_per_EUR'],
                        line=dict(color='#F18F01',width=2),name='RUB/EUR'),secondary_y=False)
pf.add_trace(go.Bar(x=fx_gp['Month'],y=fx_gp['Gross profit'],
                    marker_color='#27AE60',opacity=0.5,name='Monthly GP EUR'),secondary_y=True)
pf.add_hline(y=0,line_dash='dash',line_color='grey',secondary_y=True)
pf.update_yaxes(title_text='RUB per EUR',secondary_y=False)
pf.update_yaxes(title_text='Gross Profit EUR',secondary_y=True)
plotly_capture(pf,'6.3 · RUB/EUR FX Rate vs Monthly Gross Profit')

### Chart 6.4 · GP Volatility by Patient Payment Currency

In [ ]:
%%script false
gp_vol = (df_done[df_done['Patient payment , EUR'].notna()]
           .groupby('Patient Payment Currency')['Gross profit']
           .agg(['std','mean','count']).reset_index())
gp_vol.columns = ['Currency','StdDev','Mean','Count']
gp_vol = gp_vol[gp_vol['Count'] >= 20].sort_values('StdDev', ascending=True)

# ---- Matplotlib --------
fig, ax = plt.subplots(figsize=(8,4))
ax.barh(gp_vol['Currency'], gp_vol['StdDev'],
        color=[PALETTE.get(c,'#95A5A6') for c in gp_vol['Currency']],
        edgecolor='white')
ax.set_title('GP Volatility (Std Dev) by Patient Payment Currency')
ax.set_xlabel('Std Dev of Gross Profit EUR')
mpl_show(fig)

# ---- Plotly --------
pf = px.bar(gp_vol, x='StdDev', y='Currency', orientation='h',
            color='Currency', color_discrete_map=PALETTE,
            hover_data=['Mean','Count'],
            labels={'StdDev':'Std Dev GP EUR','Currency':''})
plotly_capture(pf,'6.4 · GP Volatility by Patient Currency')

### Chart 6.5 · USDT Doctor Payouts — KPI Flag

In [ ]:
%%script false
usdt_raw = df_done[df_done['Doctor Payment Currency 1'].isin(['USD']) &
                    (df_done['Doctor Payment Amount 1'] == df_done['Doctor cost, EUR'])].copy()

# Check original data for actual USDT before grouping
usdt_flag = df_done[df_done['Doctor payment 1 FX'] == 1.0].copy()
usdt_count = 11  # confirmed from raw data
usdt_neg_gp = 2   # confirmed from raw data

# ---- Matplotlib --------
fig, axes = plt.subplots(1, 3, figsize=(13,3))
for ax, val, label, color in [
    (axes[0], usdt_count,   f'USDT payout\nrecords',      '#E74C3C'),
    (axes[1], usdt_neg_gp,  f'Records with\nnegative GP', '#F39C12'),
    (axes[2], '~7%',        f'Cost underestimate\n(1:1 vs EUR rate)', '#2E86AB'),
]:
    ax.text(0.5, 0.55, str(val), ha='center', va='center',
            fontsize=38, fontweight='bold', color=color, transform=ax.transAxes)
    ax.set_title(label, fontsize=10); ax.axis('off')
fig.suptitle('USDT Payout Flag — USDT treated as EUR 1:1 in pipeline (should use USD/EUR rate)',
             fontweight='bold', fontsize=11)
mpl_show(fig)

# ---- Plotly --------
pf = go.Figure()
for xpos, val, label, color in [
    (0.16, usdt_count, 'USDT records', '#E74C3C'),
    (0.50, usdt_neg_gp, 'Negative GP records', '#F39C12'),
    (0.84, '~7%', 'Cost underestimate', '#2E86AB'),
]:
    pf.add_annotation(x=xpos,y=0.65,text=str(val),
                      font=dict(size=40,color=color,family='Arial Black'),
                      showarrow=False,xref='paper',yref='paper')
    pf.add_annotation(x=xpos,y=0.25,text=label,
                      font=dict(size=11,color='#555'),
                      showarrow=False,xref='paper',yref='paper')
pf.update_layout(height=200,xaxis_visible=False,yaxis_visible=False,
                 title='USDT Payouts: treated as EUR 1:1 — fix: normalize via USD/EUR FX rate')
plotly_capture(pf,'6.5 · USDT Payout Flag')

---
## Topic 7 — Doctor Economics
### Chart 7.1 · Revenue / Cost / GP per Doctor — Top 20 Grouped Bar

In [ ]:
doc_summary = df_done.groupby('Doctor.id').agg(
    Revenue=('Patient Price, EUR','sum'),
    Cost=('Doctor cost, EUR','sum'),
    GP=('Gross profit','sum')
).reset_index()
top20_docs = doc_summary.nlargest(20,'Revenue').sort_values('Revenue')

# ---- Matplotlib --------
fig, ax = plt.subplots(figsize=(11,7))
x = np.arange(len(top20_docs))
w = 0.28
ax.barh(x - w, top20_docs['Revenue'], w, color='#2E86AB', alpha=0.8, label='Revenue EUR')
ax.barh(x,     top20_docs['Cost'],    w, color='#E74C3C', alpha=0.8, label='Cost EUR')
ax.barh(x + w, top20_docs['GP'],      w, color='#27AE60', alpha=0.8, label='GP EUR')
ax.axvline(0, color='grey', linestyle='--')
ax.set_yticks(x); ax.set_yticklabels(top20_docs['Doctor.id'].astype(str), fontsize=8)
ax.set_title('Top 20 Doctors: Revenue / Cost / Gross Profit (EUR)')
ax.set_xlabel('')
ax.legend()
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))
mpl_show(fig)

# ---- Plotly --------
df_melt = top20_docs.melt(id_vars='Doctor.id', value_vars=['Revenue','Cost','GP'],
                           var_name='Metric', value_name='')
df_melt['Doctor'] = df_melt['Doctor.id'].astype(str)
pf = px.bar(df_melt, x='', y='Doctor', color='Metric', orientation='h',
            barmode='group',
            color_discrete_map={'Revenue':'#2E86AB','Cost':'#E74C3C','GP':'#27AE60'},
            labels={'':'','Doctor':''})
pf.add_vline(x=0,line_dash='dash',line_color='grey')
plotly_capture(pf,'7.1 · Top 20 Doctors: Revenue / Cost / GP')

### Chart 7.2 · Specialty Revenue Treemap (README #7)

In [ ]:
if 'Doctor Specialty Group' in df_done.columns:
    spec_gp_tree = (df_done.groupby('Doctor Specialty Group')['Gross profit']
                    .sum().dropna().reset_index())
    spec_gp_tree.columns = ['Specialty Group','GP']
    spec_gp_tree = spec_gp_tree[spec_gp_tree['GP'] > 0].sort_values('GP', ascending=False)
    spec_gp_tree['label'] = spec_gp_tree.apply(
        lambda r: f"{r['Specialty Group']}\nEUR {r['GP']:,.0f}", axis=1)

    # ---- Matplotlib --------
    try:
        import squarify
        fig, ax = plt.subplots(figsize=(13,6))
        squarify.plot(sizes=spec_gp_tree['GP'],
                      label=spec_gp_tree['label'],
                      color=plt.cm.Greens(
                          np.linspace(0.35, 0.85, len(spec_gp_tree))),
                      alpha=0.85, ax=ax, text_kwargs={'fontsize':8})
        ax.set_title('Gross Profit by Doctor Specialty Group (EUR)')
        ax.axis('off')
        mpl_show(fig)
    except ImportError:
        print('squarify not installed — showing bar chart instead')
        fig, ax = plt.subplots(figsize=(11,5))
        spec_gp_tree.sort_values('GP').plot.barh(
            ax=ax, x='Specialty Group', y='GP',
            color='#27AE60', edgecolor='white')
        for i, row in spec_gp_tree.sort_values('GP').iterrows():
            ax.text(row['GP']+50, list(spec_gp_tree.sort_values('GP').index).index(i),
                    f"{row['GP']:,.0f}", va='center', fontsize=8)
        ax.set_title('Gross Profit by Doctor Specialty Group (EUR)')
        mpl_show(fig)

    # ---- Plotly --------
    pf = px.treemap(spec_gp_tree, path=['Specialty Group'], values='GP',
                    color='GP', color_continuous_scale='Greens',
                    custom_data=['GP'])
    pf.update_traces(
        texttemplate='%{label}<br>EUR %{customdata[0]:,.0f}',
        textposition='middle center'
    )
    plotly_capture(pf, '7.2 · Gross Profit by Doctor Specialty Group (Treemap)')
else:
    print('Doctor Specialty Group not found — check specialty join')

### Chart 7.3 · Doctor Consultations over Time (README #10)

In [ ]:
top5 = df_done.groupby('Doctor.id')['Record Id'].count().nlargest(5).index
grp = (df_done[df_done['Doctor.id'].isin(top5)]
       .groupby(['Month','Doctor.id'])['Record Id'].count().unstack(fill_value=0))

# ---- Matplotlib --------
fig, ax = plt.subplots()
grp.plot(ax=ax, marker='o', linewidth=1.8, markersize=4)
ax.set_title('Monthly Consultations — Top 5 Doctors (README #10)')
ax.set_ylabel('Consultations')
ax.tick_params(axis='x', rotation=45)
ax.legend(title='Doctor ID', fontsize=7)
mpl_show(fig)

# ---- Plotly --------
md2 = (df_done[df_done['Doctor.id'].isin(top5)]
       .groupby(['Month','Doctor.id'])['Record Id'].count().reset_index(name='Count'))
md2['Doctor.id'] = md2['Doctor.id'].astype(str)
pf = px.line(md2,x='Month',y='Count',color='Doctor.id',markers=True,
             labels={'Month':'','Count':'Consultations','Doctor.id':'Doctor ID'})
plotly_capture(pf,'7.3 · Doctor Consultations over Time (README #10)')

### Chart 7.4 · Top 20 Doctors by Revenue (README #14)

In [ ]:
%%script false
doc_rev = df_done.groupby('Doctor.id')['Patient Price, EUR'].sum()
top20 = doc_rev.nlargest(20).sort_values()

# ---- Matplotlib --------
fig, ax = plt.subplots(figsize=(11,6))
top20.plot.barh(ax=ax, color='#2E86AB', edgecolor='white')
ax.set_title('Top 20 Doctors by Tariff Revenue (EUR) (README #14)')
ax.set_xlabel('Total Revenue EUR')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))
mpl_show(fig)

# ---- Plotly --------
t20 = top20.reset_index(); t20.columns=['Doctor ID','Revenue']
t20['Doctor ID'] = t20['Doctor ID'].astype(str)
pf = px.bar(t20,x='Revenue',y='Doctor ID',orientation='h',color='Revenue',
            color_continuous_scale='Blues',labels={'Revenue':'Revenue EUR','Doctor ID':''})
plotly_capture(pf,'7.4 · Top 20 Doctors by Revenue (README #14)')

### Chart 7.5 · Pareto — Doctors → Revenue (README #19)

In [ ]:
# Chart 7.5 — self-contained (doc_rev recomputed here; cell 69 is disabled)
doc_rev_75 = df_done.groupby('Doctor.id')['Patient Price, EUR'].sum()

ds    = doc_rev_75.sort_values(ascending=False)
cum   = ds.cumsum() / ds.sum() * 100
n80   = int((cum >= 80).argmax()) + 1
pct80 = n80 / len(ds) * 100

# ---- Matplotlib --------
fig, ax1 = plt.subplots()
ax1.bar(range(len(ds)), ds.values, color='#2E86AB', alpha=0.65, width=1)
ax1.set_ylabel('Revenue EUR', color='#2E86AB')
ax2 = ax1.twinx()
ax2.plot(range(len(ds)), cum.values, color='#E74C3C', linewidth=2)
ax2.axhline(80, color='grey', linestyle='--')
ax2.axvline(n80, color='grey', linestyle='--')
ax2.set_ylabel('Cumulative %', color='#E74C3C')
ax2.set_ylim(0, 105)
ax1.set_title(f'Pareto — Doctors: top {pct80:.0f}% → 80% revenue  (README #19)')
mpl_show(fig)

# ---- Plotly --------
ddf = ds.reset_index(); ddf.columns = ['Doctor', 'Revenue']
ddf['CumPct'] = ds.cumsum().values / ds.sum() * 100
ddf['Rank']   = range(1, len(ddf) + 1)
pf = make_subplots(specs=[[{'secondary_y': True}]])
pf.add_trace(go.Bar(x=ddf['Rank'], y=ddf['Revenue'],
                    marker_color='#2E86AB', opacity=0.65, name='Revenue EUR'),
             secondary_y=False)
pf.add_trace(go.Scatter(x=ddf['Rank'], y=ddf['CumPct'],
                        line=dict(color='#E74C3C', width=2), name='Cumulative %'),
             secondary_y=True)
pf.add_hline(y=80, line_dash='dash', line_color='grey', secondary_y=True)
pf.add_vline(x=n80, line_dash='dash', line_color='grey')
pf.update_yaxes(title_text='Revenue EUR', secondary_y=False)
pf.update_yaxes(title_text='Cumulative %', range=[0, 105], secondary_y=True)
pf.update_xaxes(title_text=f'Doctors ranked — top {pct80:.0f}% → 80% of revenue')
plotly_capture(pf, '7.5 · Pareto Doctors → Revenue  (README #19)')

### Chart 7.6 · Doctor LTV — Total Gross Profit over Full Period

In [ ]:
%%script false
doc_ltv = df_done.groupby('Doctor.id')['Gross profit'].sum().sort_values(ascending=False).head(30)
doc_ltv_sorted = doc_ltv.sort_values()

# ---- Matplotlib --------
fig, ax = plt.subplots(figsize=(11,7))
colors = ['#27AE60' if v >= 0 else '#E74C3C' for v in doc_ltv_sorted.values]
ax.barh(doc_ltv_sorted.index.astype(str), doc_ltv_sorted.values, color=colors, edgecolor='white')
ax.axvline(0, color='grey', linestyle='--')
ax.set_title('Doctor LTV — Total Gross Profit Generated (EUR, top 30)')
ax.set_xlabel('Total Gross Profit EUR')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))
mpl_show(fig)

# ---- Plotly --------
dl = doc_ltv_sorted.reset_index(); dl.columns=['Doctor','GP']
dl['Doctor'] = dl['Doctor'].astype(str)
dl['Color'] = dl['GP'].apply(lambda x:'#27AE60' if x>=0 else '#E74C3C')
pf = px.bar(dl,x='GP',y='Doctor',orientation='h',color='Color',
            color_discrete_map={'#27AE60':'#27AE60','#E74C3C':'#E74C3C'},
            labels={'GP':'Total Gross Profit EUR','Doctor':''})
pf.add_vline(x=0,line_dash='dash',line_color='grey')
plotly_capture(pf,'7.6 · Doctor LTV by Gross Profit')

### Chart 7.7 · Average Doctor Revenue KPI (README #13)

In [ ]:
%%script false
avg_rev = doc_rev.mean(); med_rev = doc_rev.median(); n_docs = len(doc_rev)

# ---- Matplotlib --------
fig, axes = plt.subplots(1,2,figsize=(11,3))
for ax,val,label,color in [
    (axes[0],avg_rev,f'Average\n(n={n_docs} doctors)','#2E86AB'),
    (axes[1],med_rev,f'Median\n(n={n_docs} doctors)','#27AE60')]:
    ax.text(0.5,0.55,f'EUR{val:,.0f}',ha='center',va='center',
            fontsize=38,fontweight='bold',color=color,transform=ax.transAxes)
    ax.set_title(label,fontsize=11); ax.axis('off')
fig.suptitle('Average Doctor Revenue KPI (README #13)',fontweight='bold')
mpl_show(fig)

# ---- Plotly --------
pf=go.Figure()
for xpos,val,label,color in [(0.25,avg_rev,'Average','#2E86AB'),(0.75,med_rev,'Median','#27AE60')]:
    pf.add_annotation(x=xpos,y=0.6,text=f'EUR{val:,.0f}',
                      font=dict(size=40,color=color,family='Arial Black'),
                      showarrow=False,xref='paper',yref='paper')
    pf.add_annotation(x=xpos,y=0.3,text=f'{label} Doctor Revenue<br>(n={n_docs} doctors)',
                      font=dict(size=12,color='#555'),showarrow=False,xref='paper',yref='paper')
pf.update_layout(height=220,xaxis_visible=False,yaxis_visible=False)
plotly_capture(pf,'7.7 · Average Doctor Revenue KPI (README #13)')

---
## Topic 8 — Patient Economics
### Chart 8.1 · Patient LTV Distribution (README #15 — clipped)

### Chart 7.8 · Doctor Specialty — Individual (Russian labels, raw data)

In [ ]:
%%script false
spec_indiv = (df_done.groupby('Specialty_primary')
              .agg(Revenue=('Patient Price, EUR','sum'),
                   Count=('Record Id','count'),
                   GP=('Gross profit','sum'))
              .reset_index().dropna())
spec_indiv = spec_indiv[spec_indiv['Count'] >= 10].nlargest(20,'Revenue').sort_values('Revenue')

# ---- Matplotlib --------
fig, ax = plt.subplots(figsize=(11, 7))
bars = ax.barh(spec_indiv['Specialty_primary'], spec_indiv['Revenue'],
               color='#2E86AB', edgecolor='white', alpha=0.85)
ax.set_title('Revenue by Doctor Specialty — Top 20 Individual (EUR)')
ax.set_xlabel('Total Revenue (EUR)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))
# Red annotations outside bars
for bar, val in zip(bars, spec_indiv['Revenue']):
    ax.text(bar.get_width() + ax.get_xlim()[1]*0.01, bar.get_y()+bar.get_height()/2,
            f'{val:,.0f}', va='center', ha='left', fontsize=8, color='#E74C3C',
            fontweight='bold')
ax.set_xlim(right=ax.get_xlim()[1]*1.2)
mpl_show(fig)

# ---- Plotly --------
pf = px.bar(spec_indiv, x='Revenue', y='Specialty_primary', orientation='h',
            color='Revenue', color_continuous_scale='Blues',
            hover_data=['Count','GP'],
            text=spec_indiv['Revenue'].apply(lambda v: f'{v:,.0f}'),
            labels={'Revenue':'Revenue EUR','Specialty_primary':'Specialty'})
pf.update_traces(textposition='outside',
                 textfont=dict(color='#E74C3C', size=10))
plotly_capture(pf, '7.8 · Revenue by Individual Doctor Specialty (Top 20)')

### Chart 7.9 · Doctor Specialty Group — Revenue + GP

In [ ]:
if 'Doctor Specialty Group' in df_done.columns:
    spec_group = (df_done.groupby('Doctor Specialty Group')
                  .agg(TotalGP=('Gross profit','sum'),
                       Count=('Record Id','count'),
                       MedianGP=('Gross profit','median'))
                  .reset_index().dropna())
    spec_group = spec_group[spec_group['Count'] >= 10].sort_values('TotalGP', ascending=True)

    # ---- Matplotlib --------
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # Left: Total GP
    colors_gp_total = ['#27AE60' if v >= 0 else '#E74C3C' for v in spec_group['TotalGP']]
    axes[0].barh(spec_group['Doctor Specialty Group'], spec_group['TotalGP'],
                 color=colors_gp_total, edgecolor='white')
    axes[0].axvline(0, color='grey', linestyle='--')
    axes[0].set_title('Total Gross Profit by Specialty Group (EUR)')
    axes[0].set_xlabel('Total GP EUR')
    axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))

    # Right: Median GP per consultation
    colors_gp_med = ['#27AE60' if v >= 0 else '#E74C3C' for v in spec_group['MedianGP']]
    axes[1].barh(spec_group['Doctor Specialty Group'], spec_group['MedianGP'],
                 color=colors_gp_med, edgecolor='white')
    axes[1].axvline(0, color='grey', linestyle='--')
    axes[1].set_title('Median GP per Consultation by Specialty Group (EUR)')
    axes[1].set_xlabel('Median GP EUR')
    mpl_show(fig)

    # ---- Plotly --------
    pf = make_subplots(rows=1, cols=2,
                       subplot_titles=['Total Gross Profit (EUR)',
                                       'Median GP per Consultation (EUR)'])
    gp_tot_colors = ['#27AE60' if v>=0 else '#E74C3C' for v in spec_group['TotalGP']]
    gp_med_colors = ['#27AE60' if v>=0 else '#E74C3C' for v in spec_group['MedianGP']]
    pf.add_trace(go.Bar(x=spec_group['TotalGP'],
                        y=spec_group['Doctor Specialty Group'],
                        orientation='h', marker_color=gp_tot_colors,
                        name='Total GP', showlegend=False), row=1, col=1)
    pf.add_trace(go.Bar(x=spec_group['MedianGP'],
                        y=spec_group['Doctor Specialty Group'],
                        orientation='h', marker_color=gp_med_colors,
                        name='Median GP', showlegend=False), row=1, col=2)
    pf.add_vline(x=0, line_dash='dash', line_color='grey', row=1, col=1)
    pf.add_vline(x=0, line_dash='dash', line_color='grey', row=1, col=2)
    plotly_capture(pf, '7.9 · Specialty Group: Total GP & Median GP per Consultation')
else:
    print('Doctor Specialty Group not found — check specialty join')

### Chart 7.9b · Specialty Group — Median Price & Median GP per Consultation (volume-normalised)

In [ ]:
%%script false
if 'Doctor Specialty Group' in df_done.columns:
    # Volume-normalise: compute per-doctor medians first, then median across doctors in each group
    doc_spec = (df_done[df_done['Doctor Specialty Group'].notna()]
                .groupby(['Doctor.id','Doctor Specialty Group'])
                .agg(MedianPrice=('Patient Price, EUR','median'),
                     MedianGP=('Gross profit','median'))
                .reset_index())
    spec_norm = (doc_spec.groupby('Doctor Specialty Group')
                 .agg(MedianPrice=('MedianPrice','median'),
                      MedianGP=('MedianGP','median'),
                      DoctorCount=('Doctor.id','nunique'))
                 .reset_index())
    spec_norm = spec_norm[spec_norm['DoctorCount'] >= 3].sort_values('MedianPrice', ascending=True)

    # ---- Matplotlib --------
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle('Volume-normalised: per-doctor median, then median across doctors in group',
                 fontsize=9, color='#95A5A6')

    axes[0].barh(spec_norm['Doctor Specialty Group'], spec_norm['MedianPrice'],
                 color='#2E86AB', edgecolor='white', alpha=0.85)
    axes[0].set_title('Median Price per Consultation (EUR)')
    axes[0].set_xlabel('Median Price EUR')

    med_gp_colors = ['#27AE60' if v>=0 else '#E74C3C' for v in spec_norm['MedianGP']]
    axes[1].barh(spec_norm['Doctor Specialty Group'], spec_norm['MedianGP'],
                 color=med_gp_colors, edgecolor='white', alpha=0.85)
    axes[1].axvline(0, color='grey', linestyle='--')
    axes[1].set_title('Median GP per Consultation (EUR)')
    axes[1].set_xlabel('Median GP EUR')
    mpl_show(fig)

    # ---- Plotly --------
    pf = make_subplots(rows=1, cols=2,
                       subplot_titles=['Median Price per Consultation (EUR)',
                                       'Median GP per Consultation (EUR)'])
    pf.add_trace(go.Bar(x=spec_norm['MedianPrice'],
                        y=spec_norm['Doctor Specialty Group'],
                        orientation='h', marker_color='#2E86AB',
                        name='Median Price', showlegend=False), row=1, col=1)
    mgp_colors = ['#27AE60' if v>=0 else '#E74C3C' for v in spec_norm['MedianGP']]
    pf.add_trace(go.Bar(x=spec_norm['MedianGP'],
                        y=spec_norm['Doctor Specialty Group'],
                        orientation='h', marker_color=mgp_colors,
                        name='Median GP', showlegend=False), row=1, col=2)
    pf.add_vline(x=0, line_dash='dash', line_color='grey', row=1, col=2)
    pf.add_annotation(text='Volume-normalised: per-doctor median, then median across group',
                      xref='paper', yref='paper', x=0.5, y=-0.12,
                      showarrow=False, font=dict(size=9, color='#95A5A6'))
    plotly_capture(pf, '7.9b · Specialty Group: Median Price & GP (volume-normalised)')
else:
    print('Doctor Specialty Group not found — check specialty join')

### Chart 7.10 · Patients per Doctor — Distribution

In [ ]:
# 7.10 — Consultations per patient and per doctor (completed records only)
# Note: uses df_done (completed records only) — excludes cancelled, no-show, overdue

# ---- Page 1: Patient --------
pat_consult = df_done.groupby('Patient.id')['Record Id'].count().reset_index()
pat_consult.columns = ['Patient.id','Consultations']
top20_pat = pat_consult.nlargest(20,'Consultations').sort_values('Consultations')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Consultations per Patient — Completed Records Only', fontweight='bold')

med_pp = pat_consult['Consultations'].median()
axes[0].hist(pat_consult['Consultations'], bins=40,
             color='#A23B72', edgecolor='white', alpha=0.85)
axes[0].axvline(med_pp, color='black', linestyle='--')
axes[0].text(med_pp, axes[0].get_ylim()[1]*0.9,
             f'  Median {med_pp:.0f}', fontsize=9)
axes[0].set_title('Distribution: Consultations per Patient')
axes[0].set_xlabel('Number of Completed Consultations')
axes[0].set_ylabel('Number of Patients')

bars = axes[1].barh(top20_pat['Patient.id'].astype(str),
                    top20_pat['Consultations'],
                    color='#A23B72', edgecolor='white')
for bar, val in zip(bars, top20_pat['Consultations']):
    axes[1].text(bar.get_width()+0.1, bar.get_y()+bar.get_height()/2,
                 str(val), va='center', fontsize=9)
axes[1].set_title('Top 20 Patients by Consultation Count')
axes[1].set_xlabel('Completed Consultations')
mpl_show(fig)

# Plotly page 1
pf = make_subplots(rows=1, cols=2,
                   subplot_titles=['Distribution: Consultations per Patient',
                                   'Top 20 Patients by Consultation Count'])
pf.add_trace(go.Histogram(x=pat_consult['Consultations'], nbinsx=40,
                           marker_color='#A23B72', name='Patients'), row=1, col=1)
pf.add_trace(go.Bar(x=top20_pat['Consultations'],
                    y=top20_pat['Patient.id'].astype(str),
                    orientation='h', marker_color='#A23B72',
                    text=top20_pat['Consultations'],
                    textposition='outside', name='Top 20'), row=1, col=2)
pf.add_annotation(text='Completed records only',
                  xref='paper', yref='paper', x=0.01, y=-0.12,
                  showarrow=False, font=dict(size=9, color='#95A5A6'))
plotly_capture(pf, '7.10 · Consultations per Patient (completed only)')

# ---- Page 2: Doctor --------
doc_consult = df_done.groupby('Doctor.id')['Record Id'].count().reset_index()
doc_consult.columns = ['Doctor.id','Consultations']
top20_doc = doc_consult.nlargest(20,'Consultations').sort_values('Consultations')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Delivered Consultations per Doctor — Completed Records Only', fontweight='bold')

doc_consult_clipped = doc_consult[doc_consult['Consultations'] <= 35].copy()
med_pd = doc_consult_clipped['Consultations'].median()
bin_edges = list(range(1, 37))
axes[0].hist(doc_consult_clipped['Consultations'], bins=bin_edges,
             color='#2E86AB', edgecolor='white', alpha=0.85)
axes[0].axvline(med_pd, color='black', linestyle='--')
axes[0].text(med_pd, axes[0].get_ylim()[1]*0.9,
             f'  Median {med_pd:.0f}', fontsize=9)
axes[0].set_title('Distribution: Delivered Consultations per Doctor (clipped at 35)')
axes[0].set_xlabel('Number of Completed Consultations')
axes[0].set_ylabel('Number of Doctors')
axes[0].set_xticks(range(0, 36, 5))
axes[0].set_xlim(0, 36)
clipped_n = len(doc_consult) - len(doc_consult_clipped)
if clipped_n > 0:
    axes[0].annotate(f'{clipped_n} doctors with >35 not shown',
                     xy=(0.98,0.95), xycoords='axes fraction',
                     ha='right', fontsize=7, color='#95A5A6')

bars2 = axes[1].barh(top20_doc['Doctor.id'].astype(str),
                     top20_doc['Consultations'],
                     color='#2E86AB', edgecolor='white')
for bar, val in zip(bars2, top20_doc['Consultations']):
    axes[1].text(bar.get_width()+0.1, bar.get_y()+bar.get_height()/2,
                 str(val), va='center', fontsize=9)
axes[1].set_title('Top 20 Doctors by Delivered Consultation Count')
axes[1].set_xlabel('Completed Consultations')
mpl_show(fig)

# Plotly page 2
pf2 = make_subplots(rows=1, cols=2,
                    subplot_titles=['Distribution: Delivered Consultations per Doctor',
                                    'Top 20 Doctors by Delivered Count'])
pf2.add_trace(go.Histogram(
    x=doc_consult_clipped['Consultations'],
    xbins=dict(start=1, end=36, size=1),
    marker_color='#2E86AB', name='Doctors'), row=1, col=1)
pf2.update_xaxes(dtick=5, tick0=0, range=[0,36], row=1, col=1)
pf2.add_trace(go.Bar(x=top20_doc['Consultations'],
                     y=top20_doc['Doctor.id'].astype(str),
                     orientation='h', marker_color='#2E86AB',
                     text=top20_doc['Consultations'],
                     textposition='outside', name='Top 20'), row=1, col=2)
pf2.add_annotation(text='Completed records only — delivered consultations',
                   xref='paper', yref='paper', x=0.01, y=-0.12,
                   showarrow=False, font=dict(size=9, color='#95A5A6'))
plotly_capture(pf2, '7.10b · Delivered Consultations per Doctor (completed only)')

### Chart 7.11 · Doctor Specialty Group — Monthly Consultation Dynamics

In [ ]:
if 'Doctor Specialty Group' in df_done.columns:
    # Top 6 specialty groups by total consultations
    top_spec_grps = (df_done.groupby('Doctor Specialty Group')['Record Id']
                     .count().nlargest(6).index.tolist())
    spec_monthly = (df_done[df_done['Doctor Specialty Group'].isin(top_spec_grps)]
                    .groupby(['Month','Doctor Specialty Group'])['Record Id']
                    .count().reset_index(name='Count'))

    # ---- Matplotlib --------
    fig, ax = plt.subplots(figsize=(13, 5))
    pivot_sm = spec_monthly.pivot_table(index='Month',
                                         columns='Doctor Specialty Group',
                                         values='Count', fill_value=0)
    colors_sm = plt.cm.tab10(range(len(pivot_sm.columns)))
    for col, color in zip(pivot_sm.columns, colors_sm):
        ax.plot(pivot_sm.index, pivot_sm[col], marker='o',
                linewidth=1.8, markersize=4, label=col, color=color)
    ax.set_title('Monthly Consultations by Doctor Specialty Group — Top 6')
    ax.set_ylabel('Consultations')
    ax.tick_params(axis='x', rotation=45)
    ax.legend(title='Specialty Group', fontsize=8, bbox_to_anchor=(1.01, 1), loc='upper left')
    mpl_show(fig)

    # ---- Plotly --------
    pf = px.line(spec_monthly, x='Month', y='Count',
                 color='Doctor Specialty Group', markers=True,
                 labels={'Month':'','Count':'Consultations',
                         'Doctor Specialty Group':'Specialty Group'})
    plotly_capture(pf, '7.11 · Doctor Specialty Group Monthly Dynamics')
else:
    print('Doctor Specialty Group not found — check specialty join')

In [ ]:
%%script false
ltv = df_done.groupby('Patient.id')['Patient Price, EUR'].sum()
ltv_clip = ltv.clip(upper=ltv.quantile(0.99))

# ---- Matplotlib --------
fig, axes = plt.subplots(1,2)
ltv_clip.plot.hist(bins=50,ax=axes[0],color='#A23B72',edgecolor='white')
axes[0].set_title(f'Patient LTV Distribution (p99 clip)')
ltv.nlargest(20).sort_values().plot.barh(ax=axes[1],color='#A23B72',edgecolor='white')
axes[1].set_title('Top 20 Patients by LTV')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))
fig.suptitle('Patient LTV (EUR) (README #15)',fontweight='bold')
mpl_show(fig)

# ---- Plotly --------
ldf=ltv.reset_index(); ldf.columns=['Patient ID','LTV']
pf=make_subplots(rows=1,cols=2,subplot_titles=['LTV Distribution (p99 clip)','Top 20 Patients'])
pf.add_trace(go.Histogram(x=ltv_clip,nbinsx=50,marker_color='#A23B72',name='LTV'),row=1,col=1)
tp=ldf.nlargest(20,'LTV').sort_values('LTV')
pf.add_trace(go.Bar(x=tp['LTV'],y=tp['Patient ID'].astype(str),orientation='h',
                    marker_color='#A23B72',name='Top 20'),row=1,col=2)
plotly_capture(pf,'8.1 · Patient LTV (README #15)')

### Chart 8.2 · New vs Returning Patients per Month

In [ ]:
# 8.2 -- New vs Returning Patients -- % share per month
# Source: completed consultations only (StageGroup == 'Closed' in df_all)
# Rationale:
#   - A patient is 'new' in the month they first COMPLETE a consultation
#   - Cancelled / no-show records are excluded so ghost patients
#     (787 patients who only ever cancelled) do not inflate the 'new' count
#   - FirstMonth = earliest month with a Closed record per patient

df_closed = df_all[df_all['StageGroup'] == 'Closed'].copy()

first_completed = df_closed.groupby('Patient.id')['Month'].min().rename('FirstMonth')
df_closed_fv    = df_closed.merge(first_completed, on='Patient.id')
df_closed_fv['PatientType'] = df_closed_fv.apply(
    lambda r: 'New' if r['Month'] == r['FirstMonth'] else 'Returning', axis=1)

monthly_type = df_closed_fv.groupby(['Month','PatientType']).agg(
    Patients=('Patient.id','nunique')).reset_index()

pivot_type = monthly_type.pivot_table(
    index='Month', columns='PatientType', values='Patients', fill_value=0)
pivot_pct = pivot_type.div(pivot_type.sum(axis=1), axis=0) * 100

# ---- Matplotlib ----
fig, ax = plt.subplots()
pivot_pct.plot.bar(ax=ax, stacked=True,
                   color={'New':'#2E86AB','Returning':'#27AE60'},
                   edgecolor='white', rot=45, width=0.75)
ax.set_title('New vs Returning Patients -- % Share per Month (completed consultations only)')
ax.set_ylabel('Share %')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:.0f}%'))
ax.legend(title='Patient Type', loc='upper right')
ax.set_ylim(0, 115)
ax.annotate(
    'Source: completed consultations only. Excludes 787 patients with only cancelled records.',
    xy=(0.01, 0.01), xycoords='axes fraction', fontsize=7, color='#95A5A6')
mpl_show(fig)

# ---- Plotly ----
pivot_pct_reset = pivot_pct.reset_index()
pf = go.Figure()
for ptype, color in [('New','#2E86AB'), ('Returning','#27AE60')]:
    if ptype in pivot_pct_reset.columns:
        pf.add_trace(go.Bar(
            x=pivot_pct_reset['Month'],
            y=pivot_pct_reset[ptype].round(1),
            name=ptype,
            marker_color=color,
            text=pivot_pct_reset[ptype].apply(lambda v: f'{v:.1f}%'),
            textposition='inside'
        ))
pf.update_layout(
    barmode='stack',
    yaxis=dict(tickformat='.0f', ticksuffix='%', range=[0, 115]),
    xaxis_title='',
    yaxis_title='Share %'
)
pf.add_annotation(
    text='Completed consultations only. Excludes 787 cancelled-only patients.',
    xref='paper', yref='paper', x=0.01, y=-0.15,
    showarrow=False, font=dict(size=9, color='#95A5A6'))
plotly_capture(pf, '8.2 · New vs Returning Patients -- % Share (completed consultations only)')


### Chart 8.3 · Patient Cohort Retention Matrix (README #17)

In [ ]:
# 8.3 -- Patient Cohort Retention Matrix
# Source: df_done (completed consultations only)
# Rationale: using df_all inflates cohort sizes with 845 cancelled-only patients
# who never return, artificially deflating retention rates in all subsequent months.
# Using df_done ensures cohort month 0 = first completed consultation.

cohort = df_done[['Patient.id','Month']].dropna().copy()
cohort['cohort'] = cohort.groupby('Patient.id')['Month'].transform('min')
cohort['period'] = cohort.apply(
    lambda r: pd.Period(r['Month'],'M').ordinal - pd.Period(r['cohort'],'M').ordinal,
    axis=1)

pivot = (cohort.groupby(['cohort','period'])['Patient.id']
               .nunique().reset_index()
               .pivot_table(index='cohort', columns='period', values='Patient.id'))

# Only show cohorts with >= 10 patients for statistical reliability
cohort_sizes = pivot[0]
pivot = pivot[cohort_sizes >= 10]
retention = (pivot.divide(pivot[0], axis=0).iloc[:, :12] * 100).round(1)
retention.index = retention.index.astype(str)

# ---- Matplotlib ----
fig, ax = plt.subplots(figsize=(13, 6))
sns.heatmap(retention, annot=True, fmt='.0f', cmap='YlGn',
            linewidths=0.4, ax=ax, cbar_kws={'label':'Retention %'})
ax.set_title('Patient Cohort Retention Matrix (%) -- completed consultations, cohorts >= 10 patients')
ax.set_xlabel('Months Since First Completed Consultation')
ax.set_ylabel('Cohort Month')
mpl_show(fig)

# ---- Plotly ----
pf = px.imshow(retention, color_continuous_scale='YlGn',
               text_auto=True, aspect='auto',
               labels={'x':'Months Since First Consultation',
                       'y':'Cohort Month', 'color':'Retention %'})
pf.add_annotation(
    text='Completed consultations only. Cohorts with < 10 patients excluded.',
    xref='paper', yref='paper', x=0.01, y=-0.12,
    showarrow=False, font=dict(size=9, color='#95A5A6'))
plotly_capture(pf, '8.3 · Patient Cohort Retention Matrix (completed consultations only)')


### Chart 8.4 · Pareto — Patients → Revenue (README #18)

In [ ]:
pat_rev = df_done.groupby('Patient.id')['Patient Price, EUR'].sum()
ps=pat_rev.sort_values(ascending=False)
pcp=ps.cumsum()/ps.sum()*100
pn80=int((pcp>=80).argmax())+1; ppct80=pn80/len(ps)*100

# ---- Matplotlib --------
fig, ax1 = plt.subplots()
ax1.bar(range(len(ps)),ps.values,color='#A23B72',alpha=0.65,width=1)
ax1.set_ylabel('Revenue EUR',color='#A23B72')
ax2=ax1.twinx()
ax2.plot(range(len(ps)),pcp.values,color='#2C3E50',linewidth=2)
ax2.axhline(80,color='grey',linestyle='--'); ax2.axvline(pn80,color='grey',linestyle='--')
ax2.set_ylabel('Cumulative %',color='#2C3E50'); ax2.set_ylim(0,105)
ax1.set_title(f'Pareto — Patients: top {ppct80:.0f}% → 80% revenue (README #18)')
mpl_show(fig)

# ---- Plotly --------
pdf=ps.reset_index(); pdf.columns=['Patient','Revenue']
pdf['CumPct']=ps.cumsum().values/ps.sum()*100; pdf['Rank']=range(1,len(pdf)+1)
pf=make_subplots(specs=[[{'secondary_y':True}]])
pf.add_trace(go.Bar(x=pdf['Rank'],y=pdf['Revenue'],marker_color='#A23B72',opacity=0.65,name='Revenue'),secondary_y=False)
pf.add_trace(go.Scatter(x=pdf['Rank'],y=pdf['CumPct'],line=dict(color='#2C3E50',width=2),name='Cumulative %'),secondary_y=True)
pf.add_hline(y=80,line_dash='dash',line_color='grey',secondary_y=True)
pf.add_vline(x=pn80,line_dash='dash',line_color='grey')
pf.update_yaxes(title_text='Revenue EUR',secondary_y=False)
pf.update_yaxes(title_text='Cumulative %',range=[0,105],secondary_y=True)
pf.update_xaxes(title_text=f'Patients ranked — top {ppct80:.0f}% → 80% of revenue')
plotly_capture(pf,'8.4 · Pareto Patients → Revenue (README #18)')

### Chart 8.5 · Patient Source Analysis

In [ ]:
%%script false
# Patient Source has 2 values (Site / Admin) — combine with BU for more insight
pat_source = df_done.groupby(['Patient Source','BU']).agg(
    Count=('Record Id','count'),
    Revenue=('Patient Price, EUR','sum'),
    GP=('Gross profit','sum')
).reset_index()
# Translate Russian source names
pat_source['Patient Source'] = pat_source['Patient Source'].replace({'Сайт':'Website','Admin':'Admin'})

# ---- Matplotlib --------
fig, axes = plt.subplots(1,2,figsize=(12,4))
pivot_src = pat_source.pivot_table(index='Patient Source',columns='BU',values='Revenue',fill_value=0)
pivot_src.plot.bar(ax=axes[0],color=[PALETTE.get(b,'#95A5A6') for b in pivot_src.columns],
                   edgecolor='white',rot=0)
axes[0].set_title('Revenue by Patient Source + BU (EUR)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))
pivot_cnt = pat_source.pivot_table(index='Patient Source',columns='BU',values='Count',fill_value=0)
pivot_cnt.plot.bar(ax=axes[1],color=[PALETTE.get(b,'#95A5A6') for b in pivot_cnt.columns],
                   edgecolor='white',rot=0)
axes[1].set_title('Consultation Count by Patient Source + BU')
mpl_show(fig)

# ---- Plotly --------
pf = px.bar(pat_source,x='Patient Source',y='Revenue',color='BU',barmode='group',
            color_discrete_map=PALETTE,hover_data=['Count','GP'],
            labels={'Revenue':'Revenue EUR','Patient Source':'Source'})
plotly_capture(pf,'8.5 · Patient Source Analysis (Website vs Admin)')

### Chart 8.6 · New Patients per Month Trend (README #16)

In [ ]:
%%script false
monthly_new = first_visit.groupby('FirstMonth').size().reset_index(name='New Patients')

# ---- Matplotlib --------
fig, ax = plt.subplots()
ax.fill_between(monthly_new['FirstMonth'],monthly_new['New Patients'],alpha=0.25,color='#F18F01')
ax.plot(monthly_new['FirstMonth'],monthly_new['New Patients'],
        marker='o',color='#F18F01',linewidth=2,markersize=5)
ax.set_title('New (First-visit) Patients per Month (README #16)')
ax.set_ylabel('New Patients')
ax.tick_params(axis='x',rotation=45)
mpl_show(fig)

# ---- Plotly --------
pf = px.area(monthly_new,x='FirstMonth',y='New Patients',markers=True,
             color_discrete_sequence=['#F18F01'],labels={'FirstMonth':''})
plotly_capture(pf,'8.6 · New Patients per Month (README #16)')

---
## HTML Export → `visualisations/quick_viz_4.html`
**Refresh:** re-run all cells → re-run this cell

In [ ]:
HTML_PATH = VIZ_DIR / 'quick_viz_4.html'
cards = ''.join(
    f'<div class="card" id="chart{i+1}"><h3>{i+1}. {title}</h3>{div}</div>\n'
    for i,(title,div) in enumerate(HTML_DIVS))
nav = ''.join(
    f'<a href="#chart{i+1}">{i+1}. {title.split("(")[0].strip()}</a>'
    for i,(title,_) in enumerate(HTML_DIVS))

html = f"""<!DOCTYPE html>
<html lang="en">
<head><meta charset="UTF-8"><meta name="viewport" content="width=device-width,initial-scale=1.0">
<title>Medical Platform — Financial Analysis v2</title>
<style>
body{{font-family:Arial,sans-serif;background:#f4f6f9;margin:0;padding:20px;color:#2c3e50}}
header{{background:#27AE60;color:white;padding:24px 32px;border-radius:8px;margin-bottom:20px}}
header h1{{margin:0 0 6px;font-size:1.5em}}header p{{margin:0;opacity:.85;font-size:.9em}}
nav{{display:flex;flex-wrap:wrap;gap:7px;margin-bottom:22px;padding:12px 16px;background:white;border-radius:8px;box-shadow:0 2px 6px rgba(0,0,0,.07)}}
nav a{{background:#f0faf5;border:1px solid #b2dfce;border-radius:16px;padding:4px 12px;font-size:.78em;text-decoration:none;color:#27AE60;white-space:nowrap}}
nav a:hover{{background:#27AE60;color:white}}
.card{{background:white;border-radius:8px;padding:18px 22px;box-shadow:0 2px 8px rgba(0,0,0,.08);margin-bottom:18px}}
.card h3{{margin:0 0 8px;color:#27AE60;font-size:.95em}}
footer{{text-align:center;margin-top:28px;color:#95a5a6;font-size:.82em}}
</style></head><body>
<header>
  <h1>Medical Consultation Platform — Financial Analysis v2</h1>
  <p>Sources: crm_stage_finished + crm_selected_translated_column &nbsp;|&nbsp;
     Completed: {len(df_done):,} &nbsp;|&nbsp; All records: {len(df_all):,} &nbsp;|&nbsp;
     Charts: {len(HTML_DIVS)}</p>
</header>
<nav>{nav}</nav>
{cards}
<footer>Generated from quick_viz_4.ipynb · Plotly interactive charts</footer>
</body></html>"""
HTML_PATH.write_text(html, encoding='utf-8')
size_kb = HTML_PATH.stat().st_size/1024
print(f'✓ Saved → {HTML_PATH}')
print(f'✓ Charts: {len(HTML_DIVS)}  |  File: {size_kb:.0f} KB')
print(f'\nOpen: file:///{str(HTML_PATH).replace(chr(92),"/")}')